In [ ]:
# First, install all required packages
!pip install rouge-score bert-score sacrebleu sentence-transformers transformers nltk torch

# Import necessary libraries
import torch
import nltk
import re
import os
import numpy as np
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu

# Download ALL necessary NLTK resources - explicitly and completely
print("Downloading NLTK resources...")
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

# Now import NLTK modules after downloading the resources
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords

# Set up cache directory for transformers
os.environ["TRANSFORMERS_CACHE"] = "./huggingface_cache"

def calculate_metrics(original_text, summary):
    """Calculate evaluation metrics for the summary"""
    # ROUGE Score
    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rouge_scores = rouge.score(original_text, summary)

    # BERTScore
    P, R, F1 = bert_score([summary], [original_text], model_type="bert-base-uncased")
    bert_f1 = F1.mean().item()

    # BLEU Score
    bleu = sacrebleu.corpus_bleu([summary], [[original_text]]).score

    return {
        "ROUGE-1": rouge_scores["rouge1"].fmeasure,
        "ROUGE-2": rouge_scores["rouge2"].fmeasure,
        "ROUGE-L": rouge_scores["rougeL"].fmeasure,
        "BERTScore-F1": bert_f1,
        "BLEU": bleu
    }

class LegalTermsProcessor:
    def __init__(self):
        self.legal_categories = {
            'contract_law': ['agreement', 'consideration', 'offer', 'acceptance', 'breach'],
            'employment_law': ['compensation', 'salary', 'benefits', 'probation'],
            'corporate_law': ['articles', 'memorandum', 'shareholder', 'merger'],
            'intellectual_property': ['patent', 'trademark', 'copyright', 'trade secret'],
            'litigation': ['plaintiff', 'defendant', 'jurisdiction', 'damages']
        }

    def identify_legal_terms(self, text):
        found_terms = {}
        for category, terms in self.legal_categories.items():
            found = [term for term in terms if term in text.lower()]
            if found:
                found_terms[category] = found
        return found_terms

class LegalPreprocessor:
    def __init__(self):
        try:
            self.model_name = "nlpaueb/legal-bert-base-uncased"
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name)
        except Exception as e:
            print(f"Error loading Legal-BERT model: {e}")
            print("Falling back to regular BERT model...")
            self.model_name = "bert-base-uncased"
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name, num_labels=2)

    def extract_legal_sections(self, text):
        try:
            sentences = sent_tokenize(text)
            inputs = self.tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
            with torch.no_grad():
                outputs = self.model(**inputs)
                scores = torch.softmax(outputs.logits, dim=1)[:, 1]
            threshold = torch.mean(scores) + torch.std(scores)
            return ' '.join([sent for sent, score in zip(sentences, scores) if score > threshold])
        except Exception as e:
            print(f"Error in extracting legal sections: {e}")
            print("Returning original text...")
            return text

class BartSummarizer:
    def __init__(self):
        try:
            self.summarizer = pipeline("summarization", model="facebook/bart-base", device=0 if torch.cuda.is_available() else -1)
        except Exception as e:
            print(f"Error loading BART model: {e}")
            print("Falling back to smaller model...")
            self.summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-6-6", device=0 if torch.cuda.is_available() else -1)
        self.legal_terms = LegalTermsProcessor()

    def summarize(self, text):
        try:
            sections = text.split("\n\n")
            legal_sections = []
            for section in sections:
                if self.legal_terms.identify_legal_terms(section) or len(legal_sections) == 0:
                    legal_sections.append(section)

            if not legal_sections:
                legal_sections = [text]

            summaries = []
            for section in legal_sections:
                if len(section.split()) < 10:  # Skip very short sections
                    continue

                # Handle potential token length issues
                if len(section.split()) > 500:
                    section = ' '.join(section.split()[:500])

                try:
                    summary = self.summarizer(section, max_length=150, min_length=30, do_sample=False)[0]['summary_text']
                    summaries.append(summary)
                except Exception as local_e:
                    print(f"Error summarizing section: {local_e}")
                    # Add a fallback - just use the first few sentences
                    first_sents = ' '.join(sent_tokenize(section)[:3])
                    if first_sents:
                        summaries.append(first_sents)

            return " ".join(summaries) if summaries else "Could not generate summary."
        except Exception as e:
            print(f"Error in BART summarizer: {e}")
            return "Error generating summary."

class EnsembleSummarizer:
    def __init__(self):
        try:
            self.summarizer = pipeline("summarization", model="facebook/bart-large-xsum")
        except Exception as e:
            print(f"Error loading BART-large model: {e}")
            print("Falling back to smaller model...")
            self.summarizer = pipeline("summarization", model="facebook/bart-base")

        try:
            self.legal_classifier = pipeline("text-classification", model="nlpaueb/legal-bert-base-uncased")
        except Exception as e:
            print(f"Error loading legal classifier: {e}")
            print("Falling back to sentiment analysis...")
            self.legal_classifier = pipeline("sentiment-analysis")

        try:
            self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
        except Exception as e:
            print(f"Error loading SentenceTransformer: {e}")
            print("Using simpler semantic comparison...")
            self.semantic_model = None

    def analyze_coherence(self, text, summary):
        try:
            if self.semantic_model:
                text_embedding = self.semantic_model.encode(text)
                summary_embedding = self.semantic_model.encode(summary)
                return util.pytorch_cos_sim(text_embedding, summary_embedding).item()
            else:
                # Fallback simple coherence
                text_words = set(text.lower().split())
                summary_words = set(summary.lower().split())
                if not summary_words:
                    return 0
                intersection = text_words.intersection(summary_words)
                return len(intersection) / len(summary_words)
        except Exception as e:
            print(f"Error calculating coherence: {e}")
            return 0.5  # Default middle value

    def generate_summary(self, text):
        try:
            if len(text.split()) < 10:
                return text  # Return original for very short texts

            # Handle potential token length issues
            if len(text.split()) > 500:
                text = ' '.join(text.split()[:500])

            summary = self.summarizer(text, max_length=500, min_length=50)[0]['summary_text']

            try:
                # Try to get legal score
                legal_score = self.legal_classifier(summary)[0]['score']
                return summary if legal_score > 0.5 else f"{summary} (Note: This summary may need legal review.)"
            except:
                # Fallback if legal scoring fails
                return summary
        except Exception as e:
            print(f"Error in ensemble summarizer: {e}")
            return "Could not refine summary."

# Combined functionality for legal document summarization and evaluation
class LegalDocumentProcessor:
    def __init__(self):
        print("Initializing Legal Document Processor...")
        self.preprocessor = LegalPreprocessor()
        print("Legal preprocessor initialized")
        self.bart_summarizer = BartSummarizer()
        print("BART summarizer initialized")
        self.ensemble_summarizer = EnsembleSummarizer()
        print("Ensemble summarizer initialized")

    def process_document(self, text):
        try:
            print("Processing document...")
            # Preprocess the text
            preprocessed_text = self.preprocessor.extract_legal_sections(text)
            print("Text preprocessed")

            # Generate initial summary
            bart_summary = self.bart_summarizer.summarize(preprocessed_text)
            print("BART summary generated")

            # Refine with ensemble approach
            refined_summary = self.ensemble_summarizer.generate_summary(bart_summary)
            print("Refined summary generated")

            # Calculate evaluation metrics
            metrics = calculate_metrics(text, refined_summary)
            print("Metrics calculated")

            # Calculate coherence
            coherence = self.ensemble_summarizer.analyze_coherence(text, refined_summary)
            print("Coherence analyzed")

            return {
                'original_text': text,
                'preprocessed_text': preprocessed_text,
                'bart_summary': bart_summary,
                'final_summary': refined_summary,
                'coherence_score': coherence,
                'evaluation_metrics': metrics
            }
        except Exception as e:
            print(f"Error processing document: {e}")
            return {
                'original_text': text,
                'preprocessed_text': text,
                'bart_summary': "Error generating summary",
                'final_summary': "Error processing document",
                'coherence_score': 0,
                'evaluation_metrics': {
                    "ROUGE-1": 0, "ROUGE-2": 0, "ROUGE-L": 0,
                    "BERTScore-F1": 0, "BLEU": 0
                }
            }

# Example usage with Colab-friendly setup
def main():
    # Sample text - in Colab you'd likely load from a file or input box
    text = """This is a sample legal document containing important contractual obligations and agreements.
    The parties hereby agree to the following terms and conditions. The Plaintiff alleges damages resulting
    from the breach of contract by the Defendant. The trademark rights are transferred according to section 3.
    Employee compensation will be provided as outlined in Appendix B."""

    try:
        print("Creating legal document processor...")
        processor = LegalDocumentProcessor()
        print("Processing document...")
        results = processor.process_document(text)

        # Pretty print results
        print("\n--- LEGAL DOCUMENT SUMMARIZATION RESULTS ---\n")
        print(f"ORIGINAL TEXT LENGTH: {len(results['original_text'])} characters")
        print(f"PREPROCESSED TEXT LENGTH: {len(results['preprocessed_text'])} characters")
        print("\nBASIC SUMMARY:")
        print(results['bart_summary'])
        print("\nREFINED SUMMARY:")
        print(results['final_summary'])
        print("\nCOHERENCE SCORE:")
        print(f"{results['coherence_score']:.4f}")
        print("\nEVALUATION METRICS:")
        for metric, value in results['evaluation_metrics'].items():
            print(f"{metric}: {value:.4f}")
    except Exception as e:
        print(f"An error occurred in main execution: {e}")

# For Colab usage, run this directly
if __name__ == "__main__":
    main()

Creating legal document processor...
Initializing Legal Document Processor...


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Legal preprocessor initialized


Device set to use cuda:0


BART summarizer initialized


...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Metrics calculated
Coherence analyzed

--- LEGAL DOCUMENT SUMMARIZATION RESULTS ---

ORIGINAL TEXT LENGTH: 381 characters
PREPROCESSED TEXT LENGTH: 381 characters

BASIC SUMMARY:
This is a sample legal document containing important contractual obligations and agreements.    The parties hereby agree to the following terms and conditions. The Plaintiff alleges damages resulting from the breach of the trademark rights by the Defendant. The Defendant is entitled to compensation for the damages arising according to section 3.1 of the contract. Â  ÂÂ Â o   Employee compensation will be provided as outlined in Appendix B.

REFINED SUMMARY:
Here is the full text of the contract signed by the plaintiff and the Defendant in a case brought by the Plaintiff against the Defendant over the use of the plaintiff's trademark in the United States, as published by the High Court in New York, USA.

COHERENCE SCORE:
0.7341

EVALUATION METRICS:
ROUGE-1: 0.3366
ROUGE-2: 0.0606
ROUGE-L: 0.2574
BERTScore-F1: 0

In [ ]:
# First, install all required packages
!pip install rouge-score bert-score sacrebleu sentence-transformers transformers nltk torch

# Import necessary libraries
import torch
import nltk
import re
import os
import numpy as np
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu
import textwrap

# Download ALL necessary NLTK resources
print("Downloading NLTK resources...")
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

# Now import NLTK modules after downloading the resources
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords

# Set up cache directory for transformers
os.environ["TRANSFORMERS_CACHE"] = "./huggingface_cache"

def calculate_metrics(original_text, summary):
    """Calculate evaluation metrics for the summary"""
    # ROUGE Score
    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rouge_scores = rouge.score(original_text, summary)

    # BERTScore
    P, R, F1 = bert_score([summary], [original_text], model_type="bert-base-uncased")
    bert_f1 = F1.mean().item()

    # BLEU Score
    bleu = sacrebleu.corpus_bleu([summary], [[original_text]]).score

    return {
        "ROUGE-1": rouge_scores["rouge1"].fmeasure,
        "ROUGE-2": rouge_scores["rouge2"].fmeasure,
        "ROUGE-L": rouge_scores["rougeL"].fmeasure,
        "BERTScore-F1": bert_f1,
        "BLEU": bleu
    }

class LegalTermsProcessor:
    def __init__(self):
        self.legal_categories = {
            'contract_law': ['agreement', 'consideration', 'offer', 'acceptance', 'breach'],
            'employment_law': ['compensation', 'salary', 'benefits', 'probation'],
            'corporate_law': ['articles', 'memorandum', 'shareholder', 'merger'],
            'intellectual_property': ['patent', 'trademark', 'copyright', 'trade secret'],
            'litigation': ['plaintiff', 'defendant', 'jurisdiction', 'damages']
        }

    def identify_legal_terms(self, text):
        found_terms = {}
        for category, terms in self.legal_categories.items():
            found = [term for term in terms if term in text.lower()]
            if found:
                found_terms[category] = found
        return found_terms

class LegalPreprocessor:
    def __init__(self):
        try:
            self.model_name = "nlpaueb/legal-bert-base-uncased"
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name)
        except Exception as e:
            print(f"Error loading Legal-BERT model: {e}")
            print("Falling back to regular BERT model...")
            self.model_name = "bert-base-uncased"
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name, num_labels=2)

    def extract_legal_sections(self, text):
        try:
            sentences = sent_tokenize(text)
            inputs = self.tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
            with torch.no_grad():
                outputs = self.model(**inputs)
                scores = torch.softmax(outputs.logits, dim=1)[:, 1]
            threshold = torch.mean(scores) + torch.std(scores)
            return ' '.join([sent for sent, score in zip(sentences, scores) if score > threshold])
        except Exception as e:
            print(f"Error in extracting legal sections: {e}")
            print("Returning original text...")
            return text

class BartSummarizer:
    def __init__(self):
        try:
            self.summarizer = pipeline("summarization", model="facebook/bart-base", device=0 if torch.cuda.is_available() else -1)
        except Exception as e:
            print(f"Error loading BART model: {e}")
            print("Falling back to smaller model...")
            self.summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-6-6", device=0 if torch.cuda.is_available() else -1)
        self.legal_terms = LegalTermsProcessor()

    def summarize(self, text):
        try:
            sections = text.split("\n\n")
            legal_sections = []
            for section in sections:
                if self.legal_terms.identify_legal_terms(section) or len(legal_sections) == 0:
                    legal_sections.append(section)

            if not legal_sections:
                legal_sections = [text]

            summaries = []
            for section in legal_sections:
                if len(section.split()) < 10:  # Skip very short sections
                    continue

                # Handle potential token length issues
                if len(section.split()) > 500:
                    section = ' '.join(section.split()[:500])

                try:
                    summary = self.summarizer(section, max_length=150, min_length=30, do_sample=False)[0]['summary_text']
                    summaries.append(summary)
                except Exception as local_e:
                    print(f"Error summarizing section: {local_e}")
                    # Add a fallback - just use the first few sentences
                    first_sents = ' '.join(sent_tokenize(section)[:3])
                    if first_sents:
                        summaries.append(first_sents)

            return " ".join(summaries) if summaries else "Could not generate summary."
        except Exception as e:
            print(f"Error in BART summarizer: {e}")
            return "Error generating summary."

class EnsembleSummarizer:
    def __init__(self):
        try:
            self.summarizer = pipeline("summarization", model="facebook/bart-large-xsum")
        except Exception as e:
            print(f"Error loading BART-large model: {e}")
            print("Falling back to smaller model...")
            self.summarizer = pipeline("summarization", model="facebook/bart-base")

        try:
            self.legal_classifier = pipeline("text-classification", model="nlpaueb/legal-bert-base-uncased")
        except Exception as e:
            print(f"Error loading legal classifier: {e}")
            print("Falling back to sentiment analysis...")
            self.legal_classifier = pipeline("sentiment-analysis")

        try:
            self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
        except Exception as e:
            print(f"Error loading SentenceTransformer: {e}")
            print("Using simpler semantic comparison...")
            self.semantic_model = None

    def analyze_coherence(self, text, summary):
        try:
            if self.semantic_model:
                text_embedding = self.semantic_model.encode(text)
                summary_embedding = self.semantic_model.encode(summary)
                return util.pytorch_cos_sim(text_embedding, summary_embedding).item()
            else:
                # Fallback simple coherence
                text_words = set(text.lower().split())
                summary_words = set(summary.lower().split())
                if not summary_words:
                    return 0
                intersection = text_words.intersection(summary_words)
                return len(intersection) / len(summary_words)
        except Exception as e:
            print(f"Error calculating coherence: {e}")
            return 0.5  # Default middle value

    def generate_summary(self, text):
        try:
            if len(text.split()) < 10:
                return text  # Return original for very short texts

            # Handle potential token length issues
            if len(text.split()) > 500:
                text = ' '.join(text.split()[:500])

            summary = self.summarizer(text, max_length=500, min_length=50)[0]['summary_text']

            try:
                # Try to get legal score
                legal_score = self.legal_classifier(summary)[0]['score']
                return summary if legal_score > 0.5 else f"{summary} (Note: This summary may need legal review.)"
            except:
                # Fallback if legal scoring fails
                return summary
        except Exception as e:
            print(f"Error in ensemble summarizer: {e}")
            return "Could not refine summary."

# Combined functionality for legal document summarization and evaluation
class LegalDocumentProcessor:
    def __init__(self):
        print("Initializing Legal Document Processor...")
        self.preprocessor = LegalPreprocessor()
        print("Legal preprocessor initialized")
        self.bart_summarizer = BartSummarizer()
        print("BART summarizer initialized")
        self.ensemble_summarizer = EnsembleSummarizer()
        print("Ensemble summarizer initialized")

    def process_document(self, text):
        try:
            print("Processing document...")
            # Preprocess the text
            preprocessed_text = self.preprocessor.extract_legal_sections(text)
            print("Text preprocessed")

            # Generate initial summary
            bart_summary = self.bart_summarizer.summarize(preprocessed_text)
            print("BART summary generated")

            # Refine with ensemble approach
            refined_summary = self.ensemble_summarizer.generate_summary(bart_summary)
            print("Refined summary generated")

            # Calculate evaluation metrics
            metrics = calculate_metrics(text, refined_summary)
            print("Metrics calculated")

            # Calculate coherence
            coherence = self.ensemble_summarizer.analyze_coherence(text, refined_summary)
            print("Coherence analyzed")

            return {
                'original_text': text,
                'preprocessed_text': preprocessed_text,
                'bart_summary': bart_summary,
                'final_summary': refined_summary,
                'coherence_score': coherence,
                'evaluation_metrics': metrics
            }
        except Exception as e:
            print(f"Error processing document: {e}")
            return {
                'original_text': text,
                'preprocessed_text': text,
                'bart_summary': "Error generating summary",
                'final_summary': "Error processing document",
                'coherence_score': 0,
                'evaluation_metrics': {
                    "ROUGE-1": 0, "ROUGE-2": 0, "ROUGE-L": 0,
                    "BERTScore-F1": 0, "BLEU": 0
                }
            }

# Simple function to get user input (without using google.colab.forms)
def get_user_input():
    sample_text = """This is a sample legal document containing important contractual obligations and agreements.
    The parties hereby agree to the following terms and conditions. The Plaintiff alleges damages resulting
    from the breach of contract by the Defendant. The trademark rights are transferred according to section 3.
    Employee compensation will be provided as outlined in Appendix B."""

    print("======================================================")
    print("LEGAL DOCUMENT SUMMARIZER AND EVALUATOR")
    print("======================================================")
    print("\nPlease enter the legal text you want to summarize below.")
    print("You can also paste text from a file or enter your own text.")
    print("(Press Enter with empty input to use the sample text)")

    # Simple input method
    user_input = input("Enter your legal text (or press Enter for sample): ")

    if not user_input.strip():
        print("\nUsing sample text...")
        return sample_text

    return user_input

# Define a function for file uploading instructions
def show_file_upload_instructions():
    print("\n--- To use a text file instead of direct input ---")
    print("1. Run this code in a cell to upload a file:")
    print("   from google.colab import files")
    print("   uploaded = files.upload()")
    print("   filename = next(iter(uploaded))")
    print("   with open(filename, 'r') as f:")
    print("       legal_text = f.read()")
    print("2. Then call the process_legal_text function with your file content:")
    print("   process_legal_text(legal_text)")

# Function to process text and display results
def process_legal_text(text):
    try:
        print("\nCreating legal document processor...")
        processor = LegalDocumentProcessor()
        print("Processing document...")
        results = processor.process_document(text)

        # Pretty print results
        print("\n======================================================")
        print("LEGAL DOCUMENT SUMMARIZATION RESULTS")
        print("======================================================\n")

        print(f"ORIGINAL TEXT ({len(results['original_text'])} chars, ~{len(results['original_text'].split())} words):")
        print("-" * 60)
        # Print wrapped text for better readability
        print(textwrap.fill(results['original_text'][:500], width=80))
        if len(results['original_text']) > 500:
            print("...(truncated for display)...")

        print("\nPREPROCESSED TEXT:")
        print("-" * 60)
        preprocessed_preview = results['preprocessed_text'][:300]
        print(textwrap.fill(preprocessed_preview, width=80))
        if len(results['preprocessed_text']) > 300:
            print("...(truncated for display)...")

        print("\nBASIC SUMMARY:")
        print("-" * 60)
        print(textwrap.fill(results['bart_summary'], width=80))

        print("\nREFINED SUMMARY:")
        print("-" * 60)
        print(textwrap.fill(results['final_summary'], width=80))

        print("\nEVALUATION METRICS:")
        print("-" * 60)
        print(f"COHERENCE SCORE: {results['coherence_score']:.4f}")
        for metric, value in results['evaluation_metrics'].items():
            print(f"{metric}: {value:.4f}")

        print("\nDone! You can now run the cell again with different text if needed.")

        return results

    except Exception as e:
        print(f"An error occurred in execution: {e}")
        return None

# Main function
def main():
    # Get text from user
    legal_text = get_user_input()

    # Show file upload instructions
    show_file_upload_instructions()

    # Process the text
    process_legal_text(legal_text)

# For Colab usage, run this directly
if __name__ == "__main__":
    main()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


LEGAL DOCUMENT SUMMARIZER AND EVALUATOR

Please enter the legal text you want to summarize below.
You can also paste text from a file or enter your own text.
(Press Enter with empty input to use the sample text)
Enter your legal text (or press Enter for sample): This Agreement is made and entered into on this 1st day of January 2024, by and between ABC Corp, a corporation organized and existing under the laws of India, having its principal place of business at XYZ Street, Delhi (hereinafter referred to as "Company"), and John Doe, residing at 123 Main Street, Mumbai (hereinafter referred to as "Employee").  WHEREAS, the Company desires to employ the Employee and the Employee desires to accept such employment under the terms and conditions set forth in this Agreement.  NOW, THEREFORE, in consideration of the mutual covenants contained herein, the parties agree as follows:  1. **Employment Terms**: The Employee shall serve as a Software Engineer and perform duties as assigned by the Comp

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Legal preprocessor initialized


Device set to use cuda:0


BART summarizer initialized


Device set to use cuda:0
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


Ensemble summarizer initialized
Processing document...
Processing document...
Error in extracting legal sections: 
**********************************************************************
  Resource punkt_tab not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')
  
  For more information see: https://www.nltk.org/data.html

  Attempted to load tokenizers/punkt_tab/english/

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************

Returning original text...
Text preprocessed


Your max_length is set to 500, but your input_length is only 149. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=74)


BART summary generated
Refined summary generated
Metrics calculated
Coherence analyzed

LEGAL DOCUMENT SUMMARIZATION RESULTS

ORIGINAL TEXT (1247 chars, ~191 words):
------------------------------------------------------------
This Agreement is made and entered into on this 1st day of January 2024, by and
between ABC Corp, a corporation organized and existing under the laws of India,
having its principal place of business at XYZ Street, Delhi (hereinafter
referred to as "Company"), and John Doe, residing at 123 Main Street, Mumbai
(hereinafter referred to as "Employee").  WHEREAS, the Company desires to employ
the Employee and the Employee desires to accept such employment under the terms
and conditions set forth in t
...(truncated for display)...

PREPROCESSED TEXT:
------------------------------------------------------------
This Agreement is made and entered into on this 1st day of January 2024, by and
between ABC Corp, a corporation organized and existing under the laws of India,
h

In [ ]:
# First, install all required packages
!pip install rouge-score bert-score sacrebleu sentence-transformers transformers nltk torch

# Import necessary libraries
import torch
import nltk
import re
import os
import numpy as np
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu
import textwrap

# Download ALL necessary NLTK resources
print("Downloading NLTK resources...")
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

# Now import NLTK modules after downloading the resources
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords

# Set up cache directory for transformers
os.environ["TRANSFORMERS_CACHE"] = "./huggingface_cache"

def calculate_metrics(original_text, summary):
    """Calculate evaluation metrics for the summary"""
    # ROUGE Score
    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rouge_scores = rouge.score(original_text, summary)

    # BERTScore
    P, R, F1 = bert_score([summary], [original_text], model_type="bert-base-uncased")
    bert_f1 = F1.mean().item()

    # BLEU Score
    bleu = sacrebleu.corpus_bleu([summary], [[original_text]]).score

    return {
        "ROUGE-1": rouge_scores["rouge1"].fmeasure,
        "ROUGE-2": rouge_scores["rouge2"].fmeasure,
        "ROUGE-L": rouge_scores["rougeL"].fmeasure,
        "BERTScore-F1": bert_f1,
        "BLEU": bleu
    }

class LegalTermsProcessor:
    def __init__(self):
        self.legal_categories = {
            'contract_law': ['agreement', 'consideration', 'offer', 'acceptance', 'breach', 'contract', 'covenant', 'warranty', 'indemnity', 'liability'],
            'employment_law': ['compensation', 'salary', 'benefits', 'probation', 'termination', 'severance', 'confidentiality', 'non-compete', 'overtime', 'leave'],
            'corporate_law': ['articles', 'memorandum', 'shareholder', 'merger', 'acquisition', 'incorporation', 'dissolution', 'dividend', 'voting', 'securities'],
            'intellectual_property': ['patent', 'trademark', 'copyright', 'trade secret', 'license', 'royalty', 'infringement', 'assignment', 'inventor', 'registration'],
            'litigation': ['plaintiff', 'defendant', 'jurisdiction', 'damages', 'complaint', 'discovery', 'verdict', 'judgment', 'appeal', 'settlement'],
            'real_estate': ['lease', 'landlord', 'tenant', 'property', 'easement', 'mortgage', 'deed', 'conveyance', 'title', 'zoning'],
            'regulatory': ['compliance', 'regulation', 'statute', 'ordinance', 'permit', 'license', 'audit', 'enforcement', 'penalty', 'exemption']
        }

    def identify_legal_terms(self, text):
        found_terms = {}
        for category, terms in self.legal_categories.items():
            found = [term for term in terms if term in text.lower()]
            if found:
                found_terms[category] = found
        return found_terms

class LegalPreprocessor:
    def __init__(self):
        try:
            self.model_name = "nlpaueb/legal-bert-base-uncased"
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name)
        except Exception as e:
            print(f"Error loading Legal-BERT model: {e}")
            print("Falling back to regular BERT model...")
            self.model_name = "bert-base-uncased"
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name, num_labels=2)

    def extract_legal_sections(self, text):
        try:
            sentences = sent_tokenize(text)
            inputs = self.tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
            with torch.no_grad():
                outputs = self.model(**inputs)
                scores = torch.softmax(outputs.logits, dim=1)[:, 1]
            threshold = torch.mean(scores) + torch.std(scores)
            return ' '.join([sent for sent, score in zip(sentences, scores) if score > threshold])
        except Exception as e:
            print(f"Error in extracting legal sections: {e}")
            print("Returning original text...")
            return text

class BartSummarizer:
    def __init__(self):
        try:
            self.summarizer = pipeline("summarization", model="facebook/bart-base", device=0 if torch.cuda.is_available() else -1)
        except Exception as e:
            print(f"Error loading BART model: {e}")
            print("Falling back to smaller model...")
            self.summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-6-6", device=0 if torch.cuda.is_available() else -1)
        self.legal_terms = LegalTermsProcessor()

    def summarize(self, text):
        try:
            sections = text.split("\n\n")
            legal_sections = []
            for section in sections:
                if self.legal_terms.identify_legal_terms(section) or len(legal_sections) == 0:
                    legal_sections.append(section)

            if not legal_sections:
                legal_sections = [text]

            summaries = []
            for section in legal_sections:
                if len(section.split()) < 10:  # Skip very short sections
                    continue

                # Handle potential token length issues
                if len(section.split()) > 500:
                    section = ' '.join(section.split()[:500])

                try:
                    summary = self.summarizer(section, max_length=150, min_length=30, do_sample=False)[0]['summary_text']
                    summaries.append(summary)
                except Exception as local_e:
                    print(f"Error summarizing section: {local_e}")
                    # Add a fallback - just use the first few sentences
                    first_sents = ' '.join(sent_tokenize(section)[:3])
                    if first_sents:
                        summaries.append(first_sents)

            return " ".join(summaries) if summaries else "Could not generate summary."
        except Exception as e:
            print(f"Error in BART summarizer: {e}")
            return "Error generating summary."

class EnsembleSummarizer:
    def __init__(self):
        try:
            self.summarizer = pipeline("summarization", model="facebook/bart-large-xsum")
        except Exception as e:
            print(f"Error loading BART-large model: {e}")
            print("Falling back to smaller model...")
            self.summarizer = pipeline("summarization", model="facebook/bart-base")

        try:
            self.legal_classifier = pipeline("text-classification", model="nlpaueb/legal-bert-base-uncased")
        except Exception as e:
            print(f"Error loading legal classifier: {e}")
            print("Falling back to sentiment analysis...")
            self.legal_classifier = pipeline("sentiment-analysis")

        try:
            self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
        except Exception as e:
            print(f"Error loading SentenceTransformer: {e}")
            print("Using simpler semantic comparison...")
            self.semantic_model = None

    def analyze_coherence(self, text, summary):
        try:
            if self.semantic_model:
                text_embedding = self.semantic_model.encode(text)
                summary_embedding = self.semantic_model.encode(summary)
                return util.pytorch_cos_sim(text_embedding, summary_embedding).item()
            else:
                # Fallback simple coherence
                text_words = set(text.lower().split())
                summary_words = set(summary.lower().split())
                if not summary_words:
                    return 0
                intersection = text_words.intersection(summary_words)
                return len(intersection) / len(summary_words)
        except Exception as e:
            print(f"Error calculating coherence: {e}")
            return 0.5  # Default middle value

    def generate_summary(self, text):
        try:
            if len(text.split()) < 10:
                return text  # Return original for very short texts

            # Handle potential token length issues
            if len(text.split()) > 500:
                text = ' '.join(text.split()[:500])

            summary = self.summarizer(text, max_length=500, min_length=50)[0]['summary_text']

            try:
                # Try to get legal score
                legal_score = self.legal_classifier(summary)[0]['score']
                return summary if legal_score > 0.5 else f"{summary} (Note: This summary may need legal review.)"
            except:
                # Fallback if legal scoring fails
                return summary
        except Exception as e:
            print(f"Error in ensemble summarizer: {e}")
            return "Could not refine summary."

# New class for legal compliance checking
class LegalComplianceChecker:
    def __init__(self):
        print("Initializing Legal Compliance Checker...")
        self.terms_processor = LegalTermsProcessor()
        try:
            # Try to load legal-specific model
            self.model = SentenceTransformer('all-mpnet-base-v2')  # Using a powerful model for semantic comparison
            print("Using MPNet for semantic compliance checking")
        except Exception as e:
            print(f"Error loading SentenceTransformer model: {e}")
            print("Falling back to basic term checking...")
            self.model = None

    def check_compliance(self, original_text, summary):
        # Extract legal terms from both texts
        original_terms = self.terms_processor.identify_legal_terms(original_text)
        summary_terms = self.terms_processor.identify_legal_terms(summary)

        # Calculate term preservation
        all_original_terms = []
        for terms in original_terms.values():
            all_original_terms.extend(terms)

        all_summary_terms = []
        for terms in summary_terms.values():
            all_summary_terms.extend(terms)

        # Remove duplicates
        all_original_terms = list(set(all_original_terms))
        all_summary_terms = list(set(all_summary_terms))

        # Calculate compliance metrics
        if not all_original_terms:
            term_preservation = 1.0  # No terms to preserve
        else:
            preserved_terms = [term for term in all_original_terms if term in summary.lower()]
            term_preservation = len(preserved_terms) / len(all_original_terms)

        # Calculate semantic preservation if possible
        semantic_preservation = 0.0
        if self.model and all_original_terms:
            try:
                # Get embeddings for original terms and their mentions in summary
                orig_embeddings = self.model.encode([f" {term} " for term in all_original_terms])

                # For each original term, find its best semantic match in the summary
                term_scores = []
                summary_sentences = sent_tokenize(summary)
                summary_embeddings = self.model.encode(summary_sentences)

                for term_embedding in orig_embeddings:
                    similarities = util.pytorch_cos_sim(term_embedding, summary_embeddings)
                    best_score = similarities.max().item()
                    term_scores.append(best_score)

                semantic_preservation = sum(term_scores) / len(term_scores) if term_scores else 0.0
            except Exception as e:
                print(f"Error calculating semantic preservation: {e}")
                semantic_preservation = 0.5  # Default value

        # Calculate overall compliance score
        if self.model:
            compliance_score = 0.4 * term_preservation + 0.6 * semantic_preservation
        else:
            compliance_score = term_preservation

        # Generate compliance report
        report = {
            'term_preservation_ratio': term_preservation,
            'semantic_preservation_score': semantic_preservation,
            'overall_compliance_score': compliance_score,
            'preserved_terms': list(set(all_original_terms) & set(all_summary_terms)),
            'missing_terms': list(set(all_original_terms) - set(all_summary_terms)),
            'compliance_level': self._get_compliance_level(compliance_score),
            'recommendations': self._generate_recommendations(compliance_score,
                                                            list(set(all_original_terms) - set(all_summary_terms)))
        }

        return report

    def _get_compliance_level(self, score):
        if score >= 0.85:
            return "HIGH"
        elif score >= 0.6:
            return "MODERATE"
        else:
            return "LOW"

    def _generate_recommendations(self, score, missing_terms):
        if score >= 0.85:
            return "The summary adequately preserves legal terms and context."

        recommendations = ["Consider revising the summary to include the following missing terms:"]
        for term in missing_terms[:5]:  # Limit to top 5 missing terms
            recommendations.append(f"- '{term}'")

        if score < 0.6:
            recommendations.append("The summary may significantly alter the legal meaning. Consider manual review.")

        return recommendations

# Enhanced Combined functionality for legal document summarization and evaluation
class LegalDocumentProcessor:
    def __init__(self):
        print("Initializing Legal Document Processor...")
        self.preprocessor = LegalPreprocessor()
        print("Legal preprocessor initialized")
        self.bart_summarizer = BartSummarizer()
        print("BART summarizer initialized")
        self.ensemble_summarizer = EnsembleSummarizer()
        print("Ensemble summarizer initialized")
        self.compliance_checker = LegalComplianceChecker()
        print("Legal compliance checker initialized")

    def process_document(self, text):
        try:
            print("Processing document...")
            # Preprocess the text
            preprocessed_text = self.preprocessor.extract_legal_sections(text)
            print("Text preprocessed")

            # Generate initial summary
            bart_summary = self.bart_summarizer.summarize(preprocessed_text)
            print("BART summary generated")

            # Refine with ensemble approach
            refined_summary = self.ensemble_summarizer.generate_summary(bart_summary)
            print("Refined summary generated")

            # Calculate evaluation metrics
            metrics = calculate_metrics(text, refined_summary)
            print("Metrics calculated")

            # Calculate coherence
            coherence = self.ensemble_summarizer.analyze_coherence(text, refined_summary)
            print("Coherence analyzed")

            # NEW: Check legal compliance
            compliance_report = self.compliance_checker.check_compliance(text, refined_summary)
            print("Legal compliance check completed")

            return {
                'original_text': text,
                'preprocessed_text': preprocessed_text,
                'bart_summary': bart_summary,
                'final_summary': refined_summary,
                'coherence_score': coherence,
                'evaluation_metrics': metrics,
                'legal_compliance': compliance_report  # NEW: Add compliance report
            }
        except Exception as e:
            print(f"Error processing document: {e}")
            return {
                'original_text': text,
                'preprocessed_text': text,
                'bart_summary': "Error generating summary",
                'final_summary': "Error processing document",
                'coherence_score': 0,
                'evaluation_metrics': {
                    "ROUGE-1": 0, "ROUGE-2": 0, "ROUGE-L": 0,
                    "BERTScore-F1": 0, "BLEU": 0
                },
                'legal_compliance': {
                    'term_preservation_ratio': 0,
                    'semantic_preservation_score': 0,
                    'overall_compliance_score': 0,
                    'preserved_terms': [],
                    'missing_terms': [],
                    'compliance_level': "ERROR",
                    'recommendations': ["Error processing legal compliance"]
                }
            }

# Simple function to get user input (without using google.colab.forms)
def get_user_input():
    sample_text = """This is a sample legal document containing important contractual obligations and agreements.
    The parties hereby agree to the following terms and conditions. The Plaintiff alleges damages resulting
    from the breach of contract by the Defendant. The trademark rights are transferred according to section 3.
    Employee compensation will be provided as outlined in Appendix B."""

    print("======================================================")
    print("LEGAL DOCUMENT SUMMARIZER AND EVALUATOR")
    print("======================================================")
    print("\nPlease enter the legal text you want to summarize below.")
    print("You can also paste text from a file or enter your own text.")
    print("(Press Enter with empty input to use the sample text)")

    # Simple input method
    user_input = input("Enter your legal text (or press Enter for sample): ")

    if not user_input.strip():
        print("\nUsing sample text...")
        return sample_text

    return user_input

# Define a function for file uploading instructions
def show_file_upload_instructions():
    print("\n--- To use a text file instead of direct input ---")
    print("1. Run this code in a cell to upload a file:")
    print("   from google.colab import files")
    print("   uploaded = files.upload()")
    print("   filename = next(iter(uploaded))")
    print("   with open(filename, 'r') as f:")
    print("       legal_text = f.read()")
    print("2. Then call the process_legal_text function with your file content:")
    print("   process_legal_text(legal_text)")

# Enhanced function to process text and display results with compliance information
def process_legal_text(text):
    try:
        print("\nCreating legal document processor...")
        processor = LegalDocumentProcessor()
        print("Processing document...")
        results = processor.process_document(text)

        # Pretty print results
        print("\n======================================================")
        print("LEGAL DOCUMENT SUMMARIZATION RESULTS")
        print("======================================================\n")

        print(f"ORIGINAL TEXT ({len(results['original_text'])} chars, ~{len(results['original_text'].split())} words):")
        print("-" * 60)
        # Print wrapped text for better readability
        print(textwrap.fill(results['original_text'][:500], width=80))
        if len(results['original_text']) > 500:
            print("...(truncated for display)...")

        print("\nPREPROCESSED TEXT:")
        print("-" * 60)
        preprocessed_preview = results['preprocessed_text'][:300]
        print(textwrap.fill(preprocessed_preview, width=80))
        if len(results['preprocessed_text']) > 300:
            print("...(truncated for display)...")

        print("\nBASIC SUMMARY:")
        print("-" * 60)
        print(textwrap.fill(results['bart_summary'], width=80))

        print("\nREFINED SUMMARY:")
        print("-" * 60)
        print(textwrap.fill(results['final_summary'], width=80))

        print("\nEVALUATION METRICS:")
        print("-" * 60)
        print(f"COHERENCE SCORE: {results['coherence_score']:.4f}")
        for metric, value in results['evaluation_metrics'].items():
            print(f"{metric}: {value:.4f}")

        # NEW: Display legal compliance information
        print("\nLEGAL COMPLIANCE EVALUATION:")
        print("-" * 60)
        compliance = results['legal_compliance']
        print(f"COMPLIANCE LEVEL: {compliance['compliance_level']}")
        print(f"Term Preservation: {compliance['term_preservation_ratio']:.4f}")
        print(f"Semantic Preservation: {compliance['semantic_preservation_score']:.4f}")
        print(f"Overall Compliance Score: {compliance['overall_compliance_score']:.4f}")

        print("\nPreserved Legal Terms:")
        if compliance['preserved_terms']:
            for term in compliance['preserved_terms']:
                print(f"- {term}")
        else:
            print("- None found")

        print("\nMissing Legal Terms:")
        if compliance['missing_terms']:
            for term in compliance['missing_terms']:
                print(f"- {term}")
        else:
            print("- None missing")

        print("\nRecommendations:")
        for rec in compliance['recommendations']:
            print(f"{rec}")

        print("\nDone! You can now run the cell again with different text if needed.")

        return results

    except Exception as e:
        print(f"An error occurred in execution: {e}")
        return None

# Main function
def main():
    # Get text from user
    legal_text = get_user_input()

    # Show file upload instructions
    show_file_upload_instructions()

    # Process the text
    process_legal_text(legal_text)

# For Colab usage, run this directly
if __name__ == "__main__":
    main()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


LEGAL DOCUMENT SUMMARIZER AND EVALUATOR

Please enter the legal text you want to summarize below.
You can also paste text from a file or enter your own text.
(Press Enter with empty input to use the sample text)
Enter your legal text (or press Enter for sample): Employment Contract  THIS EMPLOYMENT AGREEMENT (the "Agreement") is made and entered into as of this 1st day of April, 2025, by and between ABC Corporation, a company duly registered under the laws of India, having its principal place of business at Mumbai (hereinafter referred to as the "Employer"), and John Doe, an individual residing in New Delhi (hereinafter referred to as the "Employee").  1. Employment Terms 1.1 The Employee agrees to be employed as a Software Engineer and shall perform duties assigned by the Employer. 1.2 The Employee shall serve a probationary period of six (6) months.  2. Compensation & Benefits 2.1 The Employee shall receive a monthly salary of ₹80,000, subject to statutory deductions. 2.2 The Employe

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Legal preprocessor initialized


Device set to use cuda:0


BART summarizer initialized


Device set to use cuda:0
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at nlpaueb/legal-bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


Ensemble summarizer initialized
Initializing Legal Compliance Checker...
Using MPNet for semantic compliance checking
Legal compliance checker initialized
Processing document...
Processing document...
Error in extracting legal sections: 
**********************************************************************
  Resource punkt_tab not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')
  
  For more information see: https://www.nltk.org/data.html

  Attempted to load tokenizers/punkt_tab/english/

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************

Returning original text...
Text preprocessed


Your max_length is set to 500, but your input_length is only 147. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=73)


BART summary generated
Refined summary generated
Metrics calculated
Coherence analyzed
Error calculating semantic preservation: 
**********************************************************************
  Resource punkt_tab not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')
  
  For more information see: https://www.nltk.org/data.html

  Attempted to load tokenizers/punkt_tab/english/

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************

Legal compliance check completed

LEGAL DOCUMENT SUMMARIZATION RESULTS

ORIGINAL TEXT (1542 chars, ~248 words):
------------------------------------------------------------
Employment Contract  THIS EMPLOYMENT AGREEMENT (the "Agree

In [ ]:
# First, add T5 and Pegasus to the installation
!pip install rouge-score bert-score sacrebleu sentence-transformers transformers nltk torch

# Import necessary libraries
import torch
import nltk
import re
import os
import numpy as np
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu
import textwrap

# Download ALL necessary NLTK resources
print("Downloading NLTK resources...")
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

# Now import NLTK modules after downloading the resources
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords

# Set up cache directory for transformers
os.environ["TRANSFORMERS_CACHE"] = "./huggingface_cache"

def calculate_metrics(original_text, summary):
    """Calculate evaluation metrics for the summary"""
    # ROUGE Score
    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rouge_scores = rouge.score(original_text, summary)

    # BERTScore
    P, R, F1 = bert_score([summary], [original_text], model_type="bert-base-uncased")
    bert_f1 = F1.mean().item()

    # BLEU Score
    bleu = sacrebleu.corpus_bleu([summary], [[original_text]]).score

    return {
        "ROUGE-1": rouge_scores["rouge1"].fmeasure,
        "ROUGE-2": rouge_scores["rouge2"].fmeasure,
        "ROUGE-L": rouge_scores["rougeL"].fmeasure,
        "BERTScore-F1": bert_f1,
        "BLEU": bleu
    }

class LegalTermsProcessor:
    def __init__(self):
        self.legal_categories = {
            'contract_law': ['agreement', 'consideration', 'offer', 'acceptance', 'breach', 'contract', 'covenant', 'warranty', 'indemnity', 'liability'],
            'employment_law': ['compensation', 'salary', 'benefits', 'probation', 'termination', 'severance', 'confidentiality', 'non-compete', 'overtime', 'leave'],
            'corporate_law': ['articles', 'memorandum', 'shareholder', 'merger', 'acquisition', 'incorporation', 'dissolution', 'dividend', 'voting', 'securities'],
            'intellectual_property': ['patent', 'trademark', 'copyright', 'trade secret', 'license', 'royalty', 'infringement', 'assignment', 'inventor', 'registration'],
            'litigation': ['plaintiff', 'defendant', 'jurisdiction', 'damages', 'complaint', 'discovery', 'verdict', 'judgment', 'appeal', 'settlement'],
            'real_estate': ['lease', 'landlord', 'tenant', 'property', 'easement', 'mortgage', 'deed', 'conveyance', 'title', 'zoning'],
            'regulatory': ['compliance', 'regulation', 'statute', 'ordinance', 'permit', 'license', 'audit', 'enforcement', 'penalty', 'exemption']
        }

    def identify_legal_terms(self, text):
        found_terms = {}
        for category, terms in self.legal_categories.items():
            found = [term for term in terms if term in text.lower()]
            if found:
                found_terms[category] = found
        return found_terms

class LegalPreprocessor:
    def __init__(self):
        try:
            self.model_name = "nlpaueb/legal-bert-base-uncased"
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name)
        except Exception as e:
            print(f"Error loading Legal-BERT model: {e}")
            print("Falling back to regular BERT model...")
            self.model_name = "bert-base-uncased"
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name, num_labels=2)

    def extract_legal_sections(self, text):
        try:
            sentences = sent_tokenize(text)
            inputs = self.tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
            with torch.no_grad():
                outputs = self.model(**inputs)
                scores = torch.softmax(outputs.logits, dim=1)[:, 1]
            threshold = torch.mean(scores) + torch.std(scores)
            return ' '.join([sent for sent, score in zip(sentences, scores) if score > threshold])
        except Exception as e:
            print(f"Error in extracting legal sections: {e}")
            print("Returning original text...")
            return text

# NEW: Multi-Model Summarization System
class MultiModelSummarizer:
    def __init__(self):
        print("Initializing Multi-Model Summarizer...")
        # Initialize BART
        try:
            self.bart_summarizer = pipeline("summarization", model="facebook/bart-large-cnn",
                                           device=0 if torch.cuda.is_available() else -1)
            print("BART model loaded successfully")
        except Exception as e:
            print(f"Error loading BART model: {e}")
            print("Falling back to smaller BART model...")
            try:
                self.bart_summarizer = pipeline("summarization", model="facebook/bart-base",
                                               device=0 if torch.cuda.is_available() else -1)
            except Exception as e2:
                print(f"Error loading smaller BART model: {e2}")
                self.bart_summarizer = None

        # Initialize T5
        try:
            self.t5_summarizer = pipeline("summarization", model="t5-base",
                                         device=0 if torch.cuda.is_available() else -1)
            print("T5 model loaded successfully")
        except Exception as e:
            print(f"Error loading T5 model: {e}")
            print("Falling back to smaller T5 model...")
            try:
                self.t5_summarizer = pipeline("summarization", model="t5-small",
                                             device=0 if torch.cuda.is_available() else -1)
            except Exception as e2:
                print(f"Error loading smaller T5 model: {e2}")
                self.t5_summarizer = None

        # Initialize Pegasus
        try:
            self.pegasus_summarizer = pipeline("summarization", model="google/pegasus-xsum",
                                              device=0 if torch.cuda.is_available() else -1)
            print("Pegasus model loaded successfully")
        except Exception as e:
            print(f"Error loading Pegasus model: {e}")
            print("Falling back to smaller Pegasus model...")
            try:
                self.pegasus_summarizer = pipeline("summarization", model="google/pegasus-cnn_dailymail",
                                                  device=0 if torch.cuda.is_available() else -1)
            except Exception as e2:
                print(f"Error loading smaller Pegasus model: {e2}")
                self.pegasus_summarizer = None

        # For fusion
        try:
            self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
            print("Semantic model for fusion loaded successfully")
        except Exception as e:
            print(f"Error loading semantic model: {e}")
            self.semantic_model = None

        self.legal_terms = LegalTermsProcessor()

    def _preprocess_for_summarization(self, text):
        """Preprocess text for summarization by handling length limitations."""
        # For typical model token limits (around 1024 tokens)
        max_words = 500
        if len(text.split()) > max_words:
            return ' '.join(text.split()[:max_words])
        return text

    def _get_summary_from_model(self, model, text, max_length=150, min_length=30):
        """Get summary from a specific model with error handling."""
        if model is None:
            return None

        try:
            processed_text = self._preprocess_for_summarization(text)
            summary = model(processed_text, max_length=max_length, min_length=min_length,
                           do_sample=False)[0]['summary_text']
            return summary
        except Exception as e:
            print(f"Error generating summary with model: {e}")
            return None

    def _process_section(self, section):
        """Process a section of text with all available models."""
        summaries = {}

        # Get summaries from each model
        if self.bart_summarizer:
            summaries['bart'] = self._get_summary_from_model(self.bart_summarizer, section)

        if self.t5_summarizer:
            summaries['t5'] = self._get_summary_from_model(self.t5_summarizer, section)

        if self.pegasus_summarizer:
            summaries['pegasus'] = self._get_summary_from_model(self.pegasus_summarizer, section)

        # Return valid summaries
        return {k: v for k, v in summaries.items() if v is not None}

    def _score_summaries(self, original_text, summaries):
        """Score summaries based on semantic similarity to original text."""
        if not self.semantic_model or not summaries:
            # If no semantic model or summaries, return the first summary or None
            return next(iter(summaries.values())) if summaries else None

        try:
            # Encode original text
            orig_embedding = self.semantic_model.encode(original_text)

            # Score each summary
            scores = {}
            for name, summary in summaries.items():
                summary_embedding = self.semantic_model.encode(summary)
                score = util.pytorch_cos_sim(orig_embedding, summary_embedding).item()
                scores[name] = score

            # Find best summary
            best_model = max(scores, key=scores.get) if scores else None
            return summaries.get(best_model)
        except Exception as e:
            print(f"Error scoring summaries: {e}")
            # Fallback to first summary
            return next(iter(summaries.values())) if summaries else None

    def _fuse_summaries(self, original_text, summaries):
        """Fuse multiple summaries into a single coherent summary."""
        if not summaries:
            return "Could not generate summary."

        if len(summaries) == 1:
            return next(iter(summaries.values()))

        # Approach 1: Select best summary based on semantic similarity
        best_summary = self._score_summaries(original_text, summaries)
        if best_summary:
            return best_summary

        # Fallback: concatenate summaries if selection fails
        return " ".join(summaries.values())

    def summarize(self, text):
        """Summarize text using multiple models and fusion."""
        try:
            # Split text into sections
            sections = text.split("\n\n")
            sections = [s for s in sections if len(s.split()) >= 10]  # Skip very short sections

            if not sections:
                sections = [text]

            # Process each section with all models
            all_summaries = []
            for section in sections:
                # Check if section contains legal terms
                if self.legal_terms.identify_legal_terms(section) or len(all_summaries) == 0:
                    section_summaries = self._process_section(section)
                    fused_summary = self._fuse_summaries(section, section_summaries)
                    all_summaries.append(fused_summary)

            # Combine section summaries
            final_summary = " ".join(all_summaries)

            return final_summary if final_summary else "Could not generate summary."
        except Exception as e:
            print(f"Error in multi-model summarizer: {e}")
            return "Error generating summary."

class LegalComplianceChecker:
    def __init__(self):
        print("Initializing Legal Compliance Checker...")
        self.terms_processor = LegalTermsProcessor()
        try:
            # Try to load legal-specific model
            self.model = SentenceTransformer('all-mpnet-base-v2')  # Using a powerful model for semantic comparison
            print("Using MPNet for semantic compliance checking")
        except Exception as e:
            print(f"Error loading SentenceTransformer model: {e}")
            print("Falling back to basic term checking...")
            self.model = None

    def check_compliance(self, original_text, summary):
        # Extract legal terms from both texts
        original_terms = self.terms_processor.identify_legal_terms(original_text)
        summary_terms = self.terms_processor.identify_legal_terms(summary)

        # Calculate term preservation
        all_original_terms = []
        for terms in original_terms.values():
            all_original_terms.extend(terms)

        all_summary_terms = []
        for terms in summary_terms.values():
            all_summary_terms.extend(terms)

        # Remove duplicates
        all_original_terms = list(set(all_original_terms))
        all_summary_terms = list(set(all_summary_terms))

        # Calculate compliance metrics
        if not all_original_terms:
            term_preservation = 1.0  # No terms to preserve
        else:
            preserved_terms = [term for term in all_original_terms if term in summary.lower()]
            term_preservation = len(preserved_terms) / len(all_original_terms)

        # Calculate semantic preservation if possible
        semantic_preservation = 0.0
        if self.model and all_original_terms:
            try:
                # Get embeddings for original terms and their mentions in summary
                orig_embeddings = self.model.encode([f" {term} " for term in all_original_terms])

                # For each original term, find its best semantic match in the summary
                term_scores = []
                summary_sentences = sent_tokenize(summary)
                summary_embeddings = self.model.encode(summary_sentences)

                for term_embedding in orig_embeddings:
                    similarities = util.pytorch_cos_sim(term_embedding, summary_embeddings)
                    best_score = similarities.max().item()
                    term_scores.append(best_score)

                semantic_preservation = sum(term_scores) / len(term_scores) if term_scores else 0.0
            except Exception as e:
                print(f"Error calculating semantic preservation: {e}")
                semantic_preservation = 0.5  # Default value

        # Calculate overall compliance score
        if self.model:
            compliance_score = 0.4 * term_preservation + 0.6 * semantic_preservation
        else:
            compliance_score = term_preservation

        # Generate compliance report
        report = {
            'term_preservation_ratio': term_preservation,
            'semantic_preservation_score': semantic_preservation,
            'overall_compliance_score': compliance_score,
            'preserved_terms': list(set(all_original_terms) & set(all_summary_terms)),
            'missing_terms': list(set(all_original_terms) - set(all_summary_terms)),
            'compliance_level': self._get_compliance_level(compliance_score),
            'recommendations': self._generate_recommendations(compliance_score,
                                                            list(set(all_original_terms) - set(all_summary_terms)))
        }

        return report

    def _get_compliance_level(self, score):
        if score >= 0.85:
            return "HIGH"
        elif score >= 0.6:
            return "MODERATE"
        else:
            return "LOW"

    def _generate_recommendations(self, score, missing_terms):
        if score >= 0.85:
            return ["The summary adequately preserves legal terms and context."]

        recommendations = ["Consider revising the summary to include the following missing terms:"]
        for term in missing_terms[:5]:  # Limit to top 5 missing terms
            recommendations.append(f"- '{term}'")

        if score < 0.6:
            recommendations.append("The summary may significantly alter the legal meaning. Consider manual review.")

        return recommendations

# Modify the LegalDocumentProcessor to use the multi-model approach
class LegalDocumentProcessor:
    def __init__(self):
        print("Initializing Legal Document Processor...")
        self.preprocessor = LegalPreprocessor()
        print("Legal preprocessor initialized")
        self.multi_model_summarizer = MultiModelSummarizer()
        print("Multi-model summarizer initialized")
        self.compliance_checker = LegalComplianceChecker()
        print("Legal compliance checker initialized")

        # For enhancing the final summary with features from the original models
        try:
            self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
            print("Semantic model initialized")
        except Exception as e:
            print(f"Error loading semantic model: {e}")
            self.semantic_model = None

    def _enhance_summary(self, original_text, summary):
        """Enhance the summary with additional features."""
        try:
            if not self.semantic_model:
                return summary

            # Extract key sentences from original text
            original_sentences = sent_tokenize(original_text)
            summary_sentences = sent_tokenize(summary)

            # If summary is too short, try to add important sentences from original
            if len(summary_sentences) < 3 and len(original_sentences) > 5:
                # Encode sentences
                orig_embeddings = self.semantic_model.encode(original_sentences)
                summary_embeddings = self.semantic_model.encode(summary_sentences)

                # Find sentences in original that are important but not in summary
                important_sentences = []
                for i, orig_sent_emb in enumerate(orig_embeddings):
                    max_sim = max([util.pytorch_cos_sim(orig_sent_emb, summ_emb).item()
                                 for summ_emb in summary_embeddings], default=0)

                    # If sentence is not similar to any in summary but contains legal terms
                    if max_sim < 0.7 and self.compliance_checker.terms_processor.identify_legal_terms(original_sentences[i]):
                        important_sentences.append((original_sentences[i], max_sim))

                # Add up to 2 important sentences
                if important_sentences:
                    important_sentences.sort(key=lambda x: x[1])  # Sort by similarity (ascending)
                    for sent, _ in important_sentences[:2]:
                        if sent not in summary:
                            summary += " " + sent

            return summary
        except Exception as e:
            print(f"Error enhancing summary: {e}")
            return summary

    def process_document(self, text):
        try:
            print("Processing document...")
            # Preprocess the text
            preprocessed_text = self.preprocessor.extract_legal_sections(text)
            print("Text preprocessed")

            # Generate summary using the multi-model approach
            multi_model_summary = self.multi_model_summarizer.summarize(preprocessed_text)
            print("Multi-model summary generated")

            # Enhance the summary with additional features
            enhanced_summary = self._enhance_summary(text, multi_model_summary)
            print("Summary enhanced")

            # Calculate evaluation metrics
            metrics = calculate_metrics(text, enhanced_summary)
            print("Metrics calculated")

            # Calculate coherence
            coherence = 0.0
            if self.semantic_model:
                try:
                    text_embedding = self.semantic_model.encode(text)
                    summary_embedding = self.semantic_model.encode(enhanced_summary)
                    coherence = util.pytorch_cos_sim(text_embedding, summary_embedding).item()
                except Exception as e:
                    print(f"Error calculating coherence: {e}")
            print("Coherence analyzed")

            # Check legal compliance
            compliance_report = self.compliance_checker.check_compliance(text, enhanced_summary)
            print("Legal compliance check completed")

            return {
                'original_text': text,
                'preprocessed_text': preprocessed_text,
                'multi_model_summary': multi_model_summary,
                'final_summary': enhanced_summary,
                'coherence_score': coherence,
                'evaluation_metrics': metrics,
                'legal_compliance': compliance_report
            }
        except Exception as e:
            print(f"Error processing document: {e}")
            return {
                'original_text': text,
                'preprocessed_text': text,
                'multi_model_summary': "Error generating summary",
                'final_summary': "Error processing document",
                'coherence_score': 0,
                'evaluation_metrics': {
                    "ROUGE-1": 0, "ROUGE-2": 0, "ROUGE-L": 0,
                    "BERTScore-F1": 0, "BLEU": 0
                },
                'legal_compliance': {
                    'term_preservation_ratio': 0,
                    'semantic_preservation_score': 0,
                    'overall_compliance_score': 0,
                    'preserved_terms': [],
                    'missing_terms': [],
                    'compliance_level': "ERROR",
                    'recommendations': ["Error processing legal compliance"]
                }
            }

# Simple function to get user input (without using google.colab.forms)
def get_user_input():
    sample_text = """This is a sample legal document containing important contractual obligations and agreements.
    The parties hereby agree to the following terms and conditions. The Plaintiff alleges damages resulting
    from the breach of contract by the Defendant. The trademark rights are transferred according to section 3.
    Employee compensation will be provided as outlined in Appendix B."""

    print("======================================================")
    print("LEGAL DOCUMENT SUMMARIZER AND EVALUATOR")
    print("======================================================")
    print("\nPlease enter the legal text you want to summarize below.")
    print("You can also paste text from a file or enter your own text.")
    print("(Press Enter with empty input to use the sample text)")

    # Simple input method
    user_input = input("Enter your legal text (or press Enter for sample): ")

    if not user_input.strip():
        print("\nUsing sample text...")
        return sample_text

    return user_input

# Define a function for file uploading instructions
def show_file_upload_instructions():
    print("\n--- To use a text file instead of direct input ---")
    print("1. Run this code in a cell to upload a file:")
    print("   from google.colab import files")
    print("   uploaded = files.upload()")
    print("   filename = next(iter(uploaded))")
    print("   with open(filename, 'r') as f:")
    print("       legal_text = f.read()")
    print("2. Then call the process_legal_text function with your file content:")
    print("   process_legal_text(legal_text)")

# Updated function to process text and display results with multi-model approach
def process_legal_text(text):
    try:
        print("\nCreating legal document processor...")
        processor = LegalDocumentProcessor()
        print("Processing document...")
        results = processor.process_document(text)

        # Pretty print results
        print("\n======================================================")
        print("LEGAL DOCUMENT SUMMARIZATION RESULTS")
        print("======================================================\n")

        print(f"ORIGINAL TEXT ({len(results['original_text'])} chars, ~{len(results['original_text'].split())} words):")
        print("-" * 60)
        # Print wrapped text for better readability
        print(textwrap.fill(results['original_text'][:500], width=80))
        if len(results['original_text']) > 500:
            print("...(truncated for display)...")

        print("\nPREPROCESSED TEXT:")
        print("-" * 60)
        preprocessed_preview = results['preprocessed_text'][:300]
        print(textwrap.fill(preprocessed_preview, width=80))
        if len(results['preprocessed_text']) > 300:
            print("...(truncated for display)...")

        print("\nMULTI-MODEL SUMMARY:")
        print("-" * 60)
        print(textwrap.fill(results['multi_model_summary'], width=80))

        print("\nENHANCED FINAL SUMMARY:")
        print("-" * 60)
        print(textwrap.fill(results['final_summary'], width=80))

        print("\nEVALUATION METRICS:")
        print("-" * 60)
        print(f"COHERENCE SCORE: {results['coherence_score']:.4f}")
        for metric, value in results['evaluation_metrics'].items():
            print(f"{metric}: {value:.4f}")

        # Display legal compliance information
        print("\nLEGAL COMPLIANCE EVALUATION:")
        print("-" * 60)
        compliance = results['legal_compliance']
        print(f"COMPLIANCE LEVEL: {compliance['compliance_level']}")
        print(f"Term Preservation: {compliance['term_preservation_ratio']:.4f}")
        print(f"Semantic Preservation: {compliance['semantic_preservation_score']:.4f}")
        print(f"Overall Compliance Score: {compliance['overall_compliance_score']:.4f}")

        print("\nPreserved Legal Terms:")
        if compliance['preserved_terms']:
            for term in compliance['preserved_terms']:
                print(f"- {term}")
        else:
            print("- None found")

        print("\nMissing Legal Terms:")
        if compliance['missing_terms']:
            for term in compliance['missing_terms']:
                print(f"- {term}")
        else:
            print("- None missing")

        print("\nRecommendations:")
        for rec in compliance['recommendations']:
            print(f"{rec}")

        print("\nDone! You can now run the cell again with different text if needed.")

        return results

    except Exception as e:
        print(f"An error occurred in execution: {e}")
        return None

# Main function
def main():
    # Get text from user
    legal_text = get_user_input()

    # Show file upload instructions
    show_file_upload_instructions()

    # Process the text
    process_legal_text(legal_text)

# For Colab usage, run this directly
if __name__ == "__main__":
    main()

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


LEGAL DOCUMENT SUMMARIZER AND EVALUATOR

Please enter the legal text you want to summarize below.
You can also paste text from a file or enter your own text.
(Press Enter with empty input to use the sample text)
Enter your legal text (or press Enter for sample): Employment Contract  THIS EMPLOYMENT AGREEMENT (the "Agreement") is made and entered into as of this 1st day of April, 2025, by and between ABC Corporation, a company duly registered under the laws of India, having its principal place of business at Mumbai (hereinafter referred to as the "Employer"), and John Doe, an individual residing in New Delhi (hereinafter referred to as the "Employee").  1. Employment Terms 1.1 The Employee agrees to be employed as a Software Engineer and shall perform duties assigned by the Employer. 1.2 The Employee shall serve a probationary period of six (6) months.  2. Compensation & Benefits 2.1 The Employee shall receive a monthly salary of ₹80,000, subject to statutory deductions. 2.2 The Employe

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Metrics calculated
Coherence analyzed
Error calculating semantic preservation: 
**********************************************************************
  Resource punkt_tab not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')
  
  For more information see: https://www.nltk.org/data.html

  Attempted to load tokenizers/punkt_tab/english/

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************

Legal compliance check completed

LEGAL DOCUMENT SUMMARIZATION RESULTS

ORIGINAL TEXT (1542 chars, ~248 words):
------------------------------------------------------------
Employment Contract  THIS EMPLOYMENT AGREEMENT (the "Agreement") is made and
entered into as of this 1st da

In [ ]:
# Import necessary libraries
import torch
import nltk
import re
import os
import numpy as np
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu
import textwrap

# Download ALL necessary NLTK resources
print("Downloading NLTK resources...")
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

# Now import NLTK modules after downloading the resources
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords

# Set up cache directory for transformers
os.environ["TRANSFORMERS_CACHE"] = "./huggingface_cache"

def calculate_metrics(original_text, summary):
    """Calculate evaluation metrics for the summary"""
    # ROUGE Score
    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rouge_scores = rouge.score(original_text, summary)

    # BERTScore
    P, R, F1 = bert_score([summary], [original_text], model_type="bert-base-uncased")
    bert_f1 = F1.mean().item()

    # BLEU Score
    bleu = sacrebleu.corpus_bleu([summary], [[original_text]]).score

    return {
        "ROUGE-1": rouge_scores["rouge1"].fmeasure,
        "ROUGE-2": rouge_scores["rouge2"].fmeasure,
        "ROUGE-L": rouge_scores["rougeL"].fmeasure,
        "BERTScore-F1": bert_f1,
        "BLEU": bleu
    }

class LegalTermsProcessor:
    def _init_(self):
        self.legal_categories = {
            'contract_law': ['agreement', 'consideration', 'offer', 'acceptance', 'breach', 'contract', 'covenant', 'warranty', 'indemnity', 'liability'],
            'employment_law': ['compensation', 'salary', 'benefits', 'probation', 'termination', 'severance', 'confidentiality', 'non-compete', 'overtime', 'leave'],
            'corporate_law': ['articles', 'memorandum', 'shareholder', 'merger', 'acquisition', 'incorporation', 'dissolution', 'dividend', 'voting', 'securities'],
            'intellectual_property': ['patent', 'trademark', 'copyright', 'trade secret', 'license', 'royalty', 'infringement', 'assignment', 'inventor', 'registration'],
            'litigation': ['plaintiff', 'defendant', 'jurisdiction', 'damages', 'complaint', 'discovery', 'verdict', 'judgment', 'appeal', 'settlement'],
            'real_estate': ['lease', 'landlord', 'tenant', 'property', 'easement', 'mortgage', 'deed', 'conveyance', 'title', 'zoning'],
            'regulatory': ['compliance', 'regulation', 'statute', 'ordinance', 'permit', 'license', 'audit', 'enforcement', 'penalty', 'exemption']
        }

    def identify_legal_terms(self, text):
        found_terms = {}
        for category, terms in self.legal_categories.items():
            found = [term for term in terms if term in text.lower()]
            if found:
                found_terms[category] = found
        return found_terms

class LegalPreprocessor:
    def _init_(self):
        try:
            self.model_name = "nlpaueb/legal-bert-base-uncased"
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name)
        except Exception as e:
            print(f"Error loading Legal-BERT model: {e}")
            print("Falling back to regular BERT model...")
            self.model_name = "bert-base-uncased"
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name, num_labels=2)

    def extract_legal_sections(self, text):
        try:
            sentences = sent_tokenize(text)
            inputs = self.tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
            with torch.no_grad():
                outputs = self.model(**inputs)
                scores = torch.softmax(outputs.logits, dim=1)[:, 1]
            threshold = torch.mean(scores) + torch.std(scores)
            return ' '.join([sent for sent, score in zip(sentences, scores) if score > threshold])
        except Exception as e:
            print(f"Error in extracting legal sections: {e}")
            print("Returning original text...")
            return text

# NEW: Multi-Model Summarization System
class MultiModelSummarizer:
    def _init_(self):
        print("Initializing Multi-Model Summarizer...")
        self.bart_summarizer = None
        self.t5_summarizer = None
        self.pegasus_summarizer = None
        self.semantic_model = None

        # Initialize models with progress tracking
        print("\nLoading models (this may take a few minutes)...")

        # Initialize BART
        try:
            print("Loading BART model...")
            self.bart_summarizer = pipeline("summarization",
                                         model="facebook/bart-base",  # Using smaller model for faster loading
                                         device=0 if torch.cuda.is_available() else -1)
            print("✓ BART model loaded successfully")
        except Exception as e:
            print(f"✗ Error loading BART model: {e}")
            print("Continuing with other models...")

        # Initialize T5
        try:
            print("Loading T5 model...")
            self.t5_summarizer = pipeline("summarization",
                                       model="t5-small",  # Using smaller model for faster loading
                                       device=0 if torch.cuda.is_available() else -1)
            print("✓ T5 model loaded successfully")
        except Exception as e:
            print(f"✗ Error loading T5 model: {e}")
            print("Continuing with other models...")

        # Initialize Pegasus
        try:
            print("Loading Pegasus model...")
            self.pegasus_summarizer = pipeline("summarization",
                                            model="google/pegasus-cnn_dailymail",  # Using smaller model
                                            device=0 if torch.cuda.is_available() else -1)
            print("✓ Pegasus model loaded successfully")
        except Exception as e:
            print(f"✗ Error loading Pegasus model: {e}")
            print("Continuing with other models...")

        # For fusion and scoring
        try:
            print("Loading semantic model...")
            self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
            print("✓ Semantic model loaded successfully")
        except Exception as e:
            print(f"✗ Error loading semantic model: {e}")
            print("Continuing without semantic model...")

        # Check if at least one model is loaded
        if not any([self.bart_summarizer, self.t5_summarizer, self.pegasus_summarizer]):
            raise Exception("No summarization models could be loaded. Please check your internet connection and try again.")

        self.legal_terms = LegalTermsProcessor()
        print("\nMulti-Model Summarizer initialization complete!")
        print(f"Available models: BART: {'✓' if self.bart_summarizer else '✗'}, "
              f"T5: {'✓' if self.t5_summarizer else '✗'}, "
              f"Pegasus: {'✓' if self.pegasus_summarizer else '✗'}, "
              f"Semantic: {'✓' if self.semantic_model else '✗'}")

    def _preprocess_for_summarization(self, text):
        """Preprocess text for summarization by handling length limitations."""
        # For typical model token limits (around 1024 tokens)
        max_words = 500
        if len(text.split()) > max_words:
            return ' '.join(text.split()[:max_words])
        return text

    def _get_summary_from_model(self, model, text, max_length=150, min_length=30):
        """Get summary from a specific model with error handling and timeout."""
        if model is None:
            return None

        try:
            processed_text = self._preprocess_for_summarization(text)
            print(f"Generating summary with {model._class.name_}...")
            summary = model(processed_text, max_length=max_length, min_length=min_length,
                           do_sample=False)[0]['summary_text']
            print(f"✓ Summary generated successfully")
            return summary
        except Exception as e:
            print(f"✗ Error generating summary with model: {e}")
            return None

    def _process_section(self, section):
        """Process a section of text with all available models."""
        summaries = {}

        # Get summaries from each model
        if self.bart_summarizer:
            summaries['bart'] = self._get_summary_from_model(self.bart_summarizer, section)

        if self.t5_summarizer:
            summaries['t5'] = self._get_summary_from_model(self.t5_summarizer, section)

        if self.pegasus_summarizer:
            summaries['pegasus'] = self._get_summary_from_model(self.pegasus_summarizer, section)

        # Return valid summaries
        return {k: v for k, v in summaries.items() if v is not None}

    def _calculate_diversity_score(self, summaries):
        """Calculate diversity score between summaries."""
        if not self.semantic_model or len(summaries) < 2:
            return 0.0

        try:
            # Get embeddings for all summaries
            embeddings = [self.semantic_model.encode(summary) for summary in summaries.values()]

            # Calculate pairwise similarities
            similarities = []
            for i in range(len(embeddings)):
                for j in range(i + 1, len(embeddings)):
                    sim = util.pytorch_cos_sim(embeddings[i], embeddings[j]).item()
                    similarities.append(sim)

            # Diversity score is inverse of average similarity
            avg_similarity = sum(similarities) / len(similarities)
            return 1.0 - avg_similarity
        except Exception as e:
            print(f"Error calculating diversity score: {e}")
            return 0.0

    def _score_summaries(self, original_text, summaries):
        """Score summaries based on multiple criteria."""
        if not self.semantic_model or not summaries:
            return None

        try:
            # Encode original text
            orig_embedding = self.semantic_model.encode(original_text)

            # Calculate scores for each summary
            scores = {}
            for name, summary in summaries.items():
                # Semantic similarity score
                summary_embedding = self.semantic_model.encode(summary)
                semantic_score = util.pytorch_cos_sim(orig_embedding, summary_embedding).item()

                # Length score (prefer summaries of reasonable length)
                length_score = min(1.0, len(summary.split()) / 100)  # Normalize to [0,1]

                # Combined score
                combined_score = 0.7 * semantic_score + 0.3 * length_score
                scores[name] = combined_score

            return scores
        except Exception as e:
            print(f"Error scoring summaries: {e}")
            return None

    def _fuse_summaries(self, original_text, summaries):
        """Fuse multiple summaries using advanced strategies."""
        if not summaries:
            return "Could not generate summary."

        if len(summaries) == 1:
            return next(iter(summaries.values()))

        try:
            # Calculate scores for each summary
            scores = self._score_summaries(original_text, summaries)
            if not scores:
                return next(iter(summaries.values()))

            # Calculate diversity score
            diversity_score = self._calculate_diversity_score(summaries)

            # If diversity is high, use ensemble approach
            if diversity_score > 0.5:
                # Select top 2 summaries based on scores
                top_summaries = sorted(summaries.items(), key=lambda x: scores[x[0]], reverse=True)[:2]

                # Combine top summaries
                combined_summary = " ".join(summary for _, summary in top_summaries)

                # Remove duplicate sentences
                sentences = sent_tokenize(combined_summary)
                unique_sentences = []
                seen = set()
                for sent in sentences:
                    if sent not in seen:
                        seen.add(sent)
                        unique_sentences.append(sent)

                return " ".join(unique_sentences)
            else:
                # If diversity is low, use best summary
                best_model = max(scores, key=scores.get)
                return summaries[best_model]

        except Exception as e:
            print(f"Error in fusion: {e}")
            # Fallback to best scoring summary
            if scores:
                best_model = max(scores, key=scores.get)
                return summaries[best_model]
            return next(iter(summaries.values()))

    def summarize(self, text):
        """Summarize text using multiple models and fusion."""
        try:
            # Split text into sections
            sections = text.split("\n\n")
            sections = [s for s in sections if len(s.split()) >= 10]  # Skip very short sections

            if not sections:
                sections = [text]

            # Process each section with all models
            all_summaries = []
            for section in sections:
                # Check if section contains legal terms
                if self.legal_terms.identify_legal_terms(section) or len(all_summaries) == 0:
                    section_summaries = self._process_section(section)
                    fused_summary = self._fuse_summaries(section, section_summaries)
                    all_summaries.append(fused_summary)

            # Combine section summaries
            final_summary = " ".join(all_summaries)

            return final_summary if final_summary else "Could not generate summary."
        except Exception as e:
            print(f"Error in multi-model summarizer: {e}")
            return "Error generating summary."

class LegalComplianceChecker:
    def _init_(self):
        print("Initializing Legal Compliance Checker...")
        self.terms_processor = LegalTermsProcessor()
        try:
            # Try to load legal-specific model
            self.model = SentenceTransformer('all-mpnet-base-v2')  # Using a powerful model for semantic comparison
            print("Using MPNet for semantic compliance checking")
        except Exception as e:
            print(f"Error loading SentenceTransformer model: {e}")
            print("Falling back to basic term checking...")
            self.model = None

    def check_compliance(self, original_text, summary):
        # Extract legal terms from both texts
        original_terms = self.terms_processor.identify_legal_terms(original_text)
        summary_terms = self.terms_processor.identify_legal_terms(summary)

        # Calculate term preservation
        all_original_terms = []
        for terms in original_terms.values():
            all_original_terms.extend(terms)

        all_summary_terms = []
        for terms in summary_terms.values():
            all_summary_terms.extend(terms)

        # Remove duplicates
        all_original_terms = list(set(all_original_terms))
        all_summary_terms = list(set(all_summary_terms))

        # Calculate compliance metrics
        if not all_original_terms:
            term_preservation = 1.0  # No terms to preserve
        else:
            preserved_terms = [term for term in all_original_terms if term in summary.lower()]
            term_preservation = len(preserved_terms) / len(all_original_terms)

        # Calculate semantic preservation if possible
        semantic_preservation = 0.0
        if self.model and all_original_terms:
            try:
                # Get embeddings for original terms and their mentions in summary
                orig_embeddings = self.model.encode([f" {term} " for term in all_original_terms])

                # For each original term, find its best semantic match in the summary
                term_scores = []
                summary_sentences = sent_tokenize(summary)
                summary_embeddings = self.model.encode(summary_sentences)

                for term_embedding in orig_embeddings:
                    similarities = util.pytorch_cos_sim(term_embedding, summary_embeddings)
                    best_score = similarities.max().item()
                    term_scores.append(best_score)

                semantic_preservation = sum(term_scores) / len(term_scores) if term_scores else 0.0
            except Exception as e:
                print(f"Error calculating semantic preservation: {e}")
                semantic_preservation = 0.5  # Default value

        # Calculate overall compliance score
        if self.model:
            compliance_score = 0.4 * term_preservation + 0.6 * semantic_preservation
        else:
            compliance_score = term_preservation

        # Generate compliance report
        report = {
            'term_preservation_ratio': term_preservation,
            'semantic_preservation_score': semantic_preservation,
            'overall_compliance_score': compliance_score,
            'preserved_terms': list(set(all_original_terms) & set(all_summary_terms)),
            'missing_terms': list(set(all_original_terms) - set(all_summary_terms)),
            'compliance_level': self._get_compliance_level(compliance_score),
            'recommendations': self._generate_recommendations(compliance_score,
                                                            list(set(all_original_terms) - set(all_summary_terms)))
        }

        return report

    def _get_compliance_level(self, score):
        if score >= 0.85:
            return "HIGH"
        elif score >= 0.6:
            return "MODERATE"
        else:
            return "LOW"

    def _generate_recommendations(self, score, missing_terms):
        if score >= 0.85:
            return ["The summary adequately preserves legal terms and context."]

        recommendations = ["Consider revising the summary to include the following missing terms:"]
        for term in missing_terms[:5]:  # Limit to top 5 missing terms
            recommendations.append(f"- '{term}'")

        if score < 0.6:
            recommendations.append("The summary may significantly alter the legal meaning. Consider manual review.")

        return recommendations

# Modify the LegalDocumentProcessor to use the multi-model approach
class LegalDocumentProcessor:
    def _init_(self):
        print("Initializing Legal Document Processor...")
        self.preprocessor = LegalPreprocessor()
        print("Legal preprocessor initialized")
        self.multi_model_summarizer = MultiModelSummarizer()
        print("Multi-model summarizer initialized")
        self.compliance_checker = LegalComplianceChecker()
        print("Legal compliance checker initialized")

        # For enhancing the final summary with features from the original models
        try:
            self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
            print("Semantic model initialized")
        except Exception as e:
            print(f"Error loading semantic model: {e}")
            self.semantic_model = None

    def _enhance_summary(self, original_text, summary):
        """Enhance the summary with additional features."""
        try:
            if not self.semantic_model:
                return summary

            # Extract key sentences from original text
            original_sentences = sent_tokenize(original_text)
            summary_sentences = sent_tokenize(summary)

            # If summary is too short, try to add important sentences from original
            if len(summary_sentences) < 3 and len(original_sentences) > 5:
                # Encode sentences
                orig_embeddings = self.semantic_model.encode(original_sentences)
                summary_embeddings = self.semantic_model.encode(summary_sentences)

                # Find sentences in original that are important but not in summary
                important_sentences = []
                for i, orig_sent_emb in enumerate(orig_embeddings):
                    max_sim = max([util.pytorch_cos_sim(orig_sent_emb, summ_emb).item()
                                 for summ_emb in summary_embeddings], default=0)

                    # If sentence is not similar to any in summary but contains legal terms
                    if max_sim < 0.7 and self.compliance_checker.terms_processor.identify_legal_terms(original_sentences[i]):
                        important_sentences.append((original_sentences[i], max_sim))

                # Add up to 2 important sentences
                if important_sentences:
                    important_sentences.sort(key=lambda x: x[1])  # Sort by similarity (ascending)
                    for sent, _ in important_sentences[:2]:
                        if sent not in summary:
                            summary += " " + sent

            return summary
        except Exception as e:
            print(f"Error enhancing summary: {e}")
            return summary

    def process_document(self, text):
        try:
            print("Processing document...")
            # Preprocess the text
            preprocessed_text = self.preprocessor.extract_legal_sections(text)
            print("Text preprocessed")

            # Generate summary using the multi-model approach
            multi_model_summary = self.multi_model_summarizer.summarize(preprocessed_text)
            print("Multi-model summary generated")

            # Enhance the summary with additional features
            enhanced_summary = self._enhance_summary(text, multi_model_summary)
            print("Summary enhanced")

            # Calculate evaluation metrics
            metrics = calculate_metrics(text, enhanced_summary)
            print("Metrics calculated")

            # Calculate coherence
            coherence = 0.0
            if self.semantic_model:
                try:
                    text_embedding = self.semantic_model.encode(text)
                    summary_embedding = self.semantic_model.encode(enhanced_summary)
                    coherence = util.pytorch_cos_sim(text_embedding, summary_embedding).item()
                except Exception as e:
                    print(f"Error calculating coherence: {e}")
            print("Coherence analyzed")

            # Check legal compliance
            compliance_report = self.compliance_checker.check_compliance(text, enhanced_summary)
            print("Legal compliance check completed")

            return {
                'original_text': text,
                'preprocessed_text': preprocessed_text,
                'multi_model_summary': multi_model_summary,
                'final_summary': enhanced_summary,
                'coherence_score': coherence,
                'evaluation_metrics': metrics,
                'legal_compliance': compliance_report
            }
        except Exception as e:
            print(f"Error processing document: {e}")
            return {
                'original_text': text,
                'preprocessed_text': text,
                'multi_model_summary': "Error generating summary",
                'final_summary': "Error processing document",
                'coherence_score': 0,
                'evaluation_metrics': {
                    "ROUGE-1": 0, "ROUGE-2": 0, "ROUGE-L": 0,
                    "BERTScore-F1": 0, "BLEU": 0
                },
                'legal_compliance': {
                    'term_preservation_ratio': 0,
                    'semantic_preservation_score': 0,
                    'overall_compliance_score': 0,
                    'preserved_terms': [],
                    'missing_terms': [],
                    'compliance_level': "ERROR",
                    'recommendations': ["Error processing legal compliance"]
                }
            }

# Simple function to get user input (without using google.colab.forms)
def get_user_input():
    try:
        # Try to read from doc.txt
        with open('doc.txt', 'r', encoding='utf-8') as file:
            text = file.read()
            print("\nSuccessfully read text from doc.txt")
            return text
    except FileNotFoundError:
        print("\nError: doc.txt file not found!")
        print("Please create a doc.txt file with your legal text.")
        return None
    except Exception as e:
        print(f"\nError reading doc.txt: {e}")
        return None

# Define a function for file uploading instructions
def show_file_upload_instructions():
    print("\n--- To use a text file ---")
    print("1. Create a file named 'doc.txt' in the same directory")
    print("2. Paste your legal text into doc.txt")
    print("3. Run the program again")

# Updated function to process text and display results with multi-model approach
def process_legal_text(text):
    if text is None:
        print("\nNo text to process. Please ensure doc.txt exists and contains valid text.")
        return None

    try:
        print("\nCreating legal document processor...")
        processor = LegalDocumentProcessor()
        print("Processing document...")
        results = processor.process_document(text)

        # Pretty print results
        print("\n======================================================")
        print("LEGAL DOCUMENT SUMMARIZATION RESULTS")
        print("======================================================\n")

        print(f"ORIGINAL TEXT ({len(results['original_text'])} chars, ~{len(results['original_text'].split())} words):")
        print("-" * 60)
        # Print wrapped text for better readability
        print(textwrap.fill(results['original_text'][:500], width=80))
        if len(results['original_text']) > 500:
            print("...(truncated for display)...")

        print("\nPREPROCESSED TEXT:")
        print("-" * 60)
        preprocessed_preview = results['preprocessed_text'][:300]
        print(textwrap.fill(preprocessed_preview, width=80))
        if len(results['preprocessed_text']) > 300:
            print("...(truncated for display)...")

        print("\nMULTI-MODEL SUMMARY:")
        print("-" * 60)
        print(textwrap.fill(results['multi_model_summary'], width=80))

        print("\nENHANCED FINAL SUMMARY:")
        print("-" * 60)
        print(textwrap.fill(results['final_summary'], width=80))

        print("\nEVALUATION METRICS:")
        print("-" * 60)
        print(f"COHERENCE SCORE: {results['coherence_score']:.4f}")
        for metric, value in results['evaluation_metrics'].items():
            print(f"{metric}: {value:.4f}")

        # Display legal compliance information
        print("\nLEGAL COMPLIANCE EVALUATION:")
        print("-" * 60)
        compliance = results['legal_compliance']
        print(f"COMPLIANCE LEVEL: {compliance['compliance_level']}")
        print(f"Term Preservation: {compliance['term_preservation_ratio']:.4f}")
        print(f"Semantic Preservation: {compliance['semantic_preservation_score']:.4f}")
        print(f"Overall Compliance Score: {compliance['overall_compliance_score']:.4f}")

        print("\nPreserved Legal Terms:")
        if compliance['preserved_terms']:
            for term in compliance['preserved_terms']:
                print(f"- {term}")
        else:
            print("- None found")

        print("\nMissing Legal Terms:")
        if compliance['missing_terms']:
            for term in compliance['missing_terms']:
                print(f"- {term}")
        else:
            print("- None missing")

        print("\nRecommendations:")
        for rec in compliance['recommendations']:
            print(f"{rec}")

        print("\nDone! You can now run the cell again with different text if needed.")

        return results

    except Exception as e:
        print(f"An error occurred in execution: {e}")
        return None

# Main function
def main():
    # Get text from user
    legal_text = get_user_input()

    # Show file upload instructions
    show_file_upload_instructions()

    # Process the text
    process_legal_text(legal_text)

# For Colab usage, run this directly
if _name_ == "_main_":
    main()

In [ ]:
# Import necessary libraries
!pip install torch nltk transformers sentence-transformers rouge-score bert-score sacrebleu textwrap3
import torch
import nltk
import re
import os
import numpy as np
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import sacrebleu
import textwrap

# Download ALL necessary NLTK resources
print("Downloading NLTK resources...")
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')

# Now import NLTK modules after downloading the resources
from nltk.tokenize import sent_tokenize
from nltk.corpus import stopwords

# Set up cache directory for transformers
os.environ["TRANSFORMERS_CACHE"] = "./huggingface_cache"

def calculate_metrics(original_text, summary):
    """Calculate evaluation metrics for the summary"""
    # ROUGE Score
    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    rouge_scores = rouge.score(original_text, summary)

    # BERTScore
    P, R, F1 = bert_score([summary], [original_text], model_type="bert-base-uncased")
    bert_f1 = F1.mean().item()

    # BLEU Score
    bleu = sacrebleu.corpus_bleu([summary], [[original_text]]).score

    return {
        "ROUGE-1": rouge_scores["rouge1"].fmeasure,
        "ROUGE-2": rouge_scores["rouge2"].fmeasure,
        "ROUGE-L": rouge_scores["rougeL"].fmeasure,
        "BERTScore-F1": bert_f1,
        "BLEU": bleu
    }

class LegalTermsProcessor:
    def __init__(self):
        self.legal_categories = {
            'contract_law': ['agreement', 'consideration', 'offer', 'acceptance', 'breach', 'contract', 'covenant', 'warranty', 'indemnity', 'liability'],
            'employment_law': ['compensation', 'salary', 'benefits', 'probation', 'termination', 'severance', 'confidentiality', 'non-compete', 'overtime', 'leave'],
            'corporate_law': ['articles', 'memorandum', 'shareholder', 'merger', 'acquisition', 'incorporation', 'dissolution', 'dividend', 'voting', 'securities'],
            'intellectual_property': ['patent', 'trademark', 'copyright', 'trade secret', 'license', 'royalty', 'infringement', 'assignment', 'inventor', 'registration'],
            'litigation': ['plaintiff', 'defendant', 'jurisdiction', 'damages', 'complaint', 'discovery', 'verdict', 'judgment', 'appeal', 'settlement'],
            'real_estate': ['lease', 'landlord', 'tenant', 'property', 'easement', 'mortgage', 'deed', 'conveyance', 'title', 'zoning'],
            'regulatory': ['compliance', 'regulation', 'statute', 'ordinance', 'permit', 'license', 'audit', 'enforcement', 'penalty', 'exemption']
        }

    def identify_legal_terms(self, text):
        found_terms = {}
        for category, terms in self.legal_categories.items():
            found = [term for term in terms if term in text.lower()]
            if found:
                found_terms[category] = found
        return found_terms

class LegalPreprocessor:
    def __init__(self):
        try:
            self.model_name = "nlpaueb/legal-bert-base-uncased"
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name)
        except Exception as e:
            print(f"Error loading Legal-BERT model: {e}")
            print("Falling back to regular BERT model...")
            self.model_name = "bert-base-uncased"
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name, num_labels=2)

    def extract_legal_sections(self, text):
        try:
            sentences = sent_tokenize(text)
            inputs = self.tokenizer(sentences, padding=True, truncation=True, return_tensors="pt")
            with torch.no_grad():
                outputs = self.model(**inputs)
                scores = torch.softmax(outputs.logits, dim=1)[:, 1]
            threshold = torch.mean(scores) + torch.std(scores)
            return ' '.join([sent for sent, score in zip(sentences, scores) if score > threshold])
        except Exception as e:
            print(f"Error in extracting legal sections: {e}")
            print("Returning original text...")
            return text

# NEW: Multi-Model Summarization System
class MultiModelSummarizer:
    def __init__(self):
        print("Initializing Multi-Model Summarizer...")
        self.bart_summarizer = None
        self.t5_summarizer = None
        self.pegasus_summarizer = None
        self.semantic_model = None

        # Initialize models with progress tracking
        print("\nLoading models (this may take a few minutes)...")

        # Initialize BART
        try:
            print("Loading BART model...")
            self.bart_summarizer = pipeline("summarization",
                                         model="facebook/bart-base",  # Using smaller model for faster loading
                                         device=0 if torch.cuda.is_available() else -1)
            print("✓ BART model loaded successfully")
        except Exception as e:
            print(f"✗ Error loading BART model: {e}")
            print("Continuing with other models...")

        # Initialize T5
        try:
            print("Loading T5 model...")
            self.t5_summarizer = pipeline("summarization",
                                       model="t5-small",  # Using smaller model for faster loading
                                       device=0 if torch.cuda.is_available() else -1)
            print("✓ T5 model loaded successfully")
        except Exception as e:
            print(f"✗ Error loading T5 model: {e}")
            print("Continuing with other models...")

        # Initialize Pegasus
        try:
            print("Loading Pegasus model...")
            self.pegasus_summarizer = pipeline("summarization",
                                            model="google/pegasus-cnn_dailymail",  # Using smaller model
                                            device=0 if torch.cuda.is_available() else -1)
            print("✓ Pegasus model loaded successfully")
        except Exception as e:
            print(f"✗ Error loading Pegasus model: {e}")
            print("Continuing with other models...")

        # For fusion and scoring
        try:
            print("Loading semantic model...")
            self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
            print("✓ Semantic model loaded successfully")
        except Exception as e:
            print(f"✗ Error loading semantic model: {e}")
            print("Continuing without semantic model...")

        # Check if at least one model is loaded
        if not any([self.bart_summarizer, self.t5_summarizer, self.pegasus_summarizer]):
            raise Exception("No summarization models could be loaded. Please check your internet connection and try again.")

        self.legal_terms = LegalTermsProcessor()
        print("\nMulti-Model Summarizer initialization complete!")
        print(f"Available models: BART: {'✓' if self.bart_summarizer else '✗'}, "
              f"T5: {'✓' if self.t5_summarizer else '✗'}, "
              f"Pegasus: {'✓' if self.pegasus_summarizer else '✗'}, "
              f"Semantic: {'✓' if self.semantic_model else '✗'}")

    def _preprocess_for_summarization(self, text):
        """Preprocess text for summarization by handling length limitations."""
        # For typical model token limits (around 1024 tokens)
        max_words = 500
        if len(text.split()) > max_words:
            return ' '.join(text.split()[:max_words])
        return text

    def _get_summary_from_model(self, model, text, max_length=150, min_length=30):
        """Get summary from a specific model with error handling and timeout."""
        if model is None:
            return None

        try:
            processed_text = self._preprocess_for_summarization(text)
            print(f"Generating summary with {model.__class__.__name__}...")
            summary = model(processed_text, max_length=max_length, min_length=min_length,
                           do_sample=False)[0]['summary_text']
            print(f"✓ Summary generated successfully")
            return summary
        except Exception as e:
            print(f"✗ Error generating summary with model: {e}")
            return None

    def _process_section(self, section):
        """Process a section of text with all available models."""
        summaries = {}

        # Get summaries from each model
        if self.bart_summarizer:
            summaries['bart'] = self._get_summary_from_model(self.bart_summarizer, section)

        if self.t5_summarizer:
            summaries['t5'] = self._get_summary_from_model(self.t5_summarizer, section)

        if self.pegasus_summarizer:
            summaries['pegasus'] = self._get_summary_from_model(self.pegasus_summarizer, section)

        # Return valid summaries
        return {k: v for k, v in summaries.items() if v is not None}

    def _calculate_diversity_score(self, summaries):
        """Calculate diversity score between summaries."""
        if not self.semantic_model or len(summaries) < 2:
            return 0.0

        try:
            # Get embeddings for all summaries
            embeddings = [self.semantic_model.encode(summary) for summary in summaries.values()]

            # Calculate pairwise similarities
            similarities = []
            for i in range(len(embeddings)):
                for j in range(i + 1, len(embeddings)):
                    sim = util.pytorch_cos_sim(embeddings[i], embeddings[j]).item()
                    similarities.append(sim)

            # Diversity score is inverse of average similarity
            avg_similarity = sum(similarities) / len(similarities)
            return 1.0 - avg_similarity
        except Exception as e:
            print(f"Error calculating diversity score: {e}")
            return 0.0

    def _score_summaries(self, original_text, summaries):
        """Score summaries based on multiple criteria."""
        if not self.semantic_model or not summaries:
            return None

        try:
            # Encode original text
            orig_embedding = self.semantic_model.encode(original_text)

            # Calculate scores for each summary
            scores = {}
            for name, summary in summaries.items():
                # Semantic similarity score
                summary_embedding = self.semantic_model.encode(summary)
                semantic_score = util.pytorch_cos_sim(orig_embedding, summary_embedding).item()

                # Length score (prefer summaries of reasonable length)
                length_score = min(1.0, len(summary.split()) / 100)  # Normalize to [0,1]

                # Combined score
                combined_score = 0.7 * semantic_score + 0.3 * length_score
                scores[name] = combined_score

            return scores
        except Exception as e:
            print(f"Error scoring summaries: {e}")
            return None

    def _fuse_summaries(self, original_text, summaries):
        """Fuse multiple summaries using advanced strategies."""
        if not summaries:
            return "Could not generate summary."

        if len(summaries) == 1:
            return next(iter(summaries.values()))

        try:
            # Calculate scores for each summary
            scores = self._score_summaries(original_text, summaries)
            if not scores:
                return next(iter(summaries.values()))

            # Calculate diversity score
            diversity_score = self._calculate_diversity_score(summaries)

            # If diversity is high, use ensemble approach
            if diversity_score > 0.5:
                # Select top 2 summaries based on scores
                top_summaries = sorted(summaries.items(), key=lambda x: scores[x[0]], reverse=True)[:2]

                # Combine top summaries
                combined_summary = " ".join(summary for _, summary in top_summaries)

                # Remove duplicate sentences
                sentences = sent_tokenize(combined_summary)
                unique_sentences = []
                seen = set()
                for sent in sentences:
                    if sent not in seen:
                        seen.add(sent)
                        unique_sentences.append(sent)

                return " ".join(unique_sentences)
            else:
                # If diversity is low, use best summary
                best_model = max(scores, key=scores.get)
                return summaries[best_model]

        except Exception as e:
            print(f"Error in fusion: {e}")
            # Fallback to best scoring summary
            if scores:
                best_model = max(scores, key=scores.get)
                return summaries[best_model]
            return next(iter(summaries.values()))

    def summarize(self, text):
        """Summarize text using multiple models and fusion."""
        try:
            # Split text into sections
            sections = text.split("\n\n")
            sections = [s for s in sections if len(s.split()) >= 10]  # Skip very short sections

            if not sections:
                sections = [text]

            # Process each section with all models
            all_summaries = []
            for section in sections:
                # Check if section contains legal terms
                if self.legal_terms.identify_legal_terms(section) or len(all_summaries) == 0:
                    section_summaries = self._process_section(section)
                    fused_summary = self._fuse_summaries(section, section_summaries)
                    all_summaries.append(fused_summary)

            # Combine section summaries
            final_summary = " ".join(all_summaries)

            return final_summary if final_summary else "Could not generate summary."
        except Exception as e:
            print(f"Error in multi-model summarizer: {e}")
            return "Error generating summary."

class LegalComplianceChecker:
    def __init__(self):
        print("Initializing Legal Compliance Checker...")
        self.terms_processor = LegalTermsProcessor()
        try:
            # Try to load legal-specific model
            self.model = SentenceTransformer('all-mpnet-base-v2')  # Using a powerful model for semantic comparison
            print("Using MPNet for semantic compliance checking")
        except Exception as e:
            print(f"Error loading SentenceTransformer model: {e}")
            print("Falling back to basic term checking...")
            self.model = None

    def check_compliance(self, original_text, summary):
        # Extract legal terms from both texts
        original_terms = self.terms_processor.identify_legal_terms(original_text)
        summary_terms = self.terms_processor.identify_legal_terms(summary)

        # Calculate term preservation
        all_original_terms = []
        for terms in original_terms.values():
            all_original_terms.extend(terms)

        all_summary_terms = []
        for terms in summary_terms.values():
            all_summary_terms.extend(terms)

        # Remove duplicates
        all_original_terms = list(set(all_original_terms))
        all_summary_terms = list(set(all_summary_terms))

        # Calculate compliance metrics
        if not all_original_terms:
            term_preservation = 1.0  # No terms to preserve
        else:
            preserved_terms = [term for term in all_original_terms if term in summary.lower()]
            term_preservation = len(preserved_terms) / len(all_original_terms)

        # Calculate semantic preservation if possible
        semantic_preservation = 0.0
        if self.model and all_original_terms:
            try:
                # Get embeddings for original terms and their mentions in summary
                orig_embeddings = self.model.encode([f" {term} " for term in all_original_terms])

                # For each original term, find its best semantic match in the summary
                term_scores = []
                summary_sentences = sent_tokenize(summary)
                summary_embeddings = self.model.encode(summary_sentences)

                for term_embedding in orig_embeddings:
                    similarities = util.pytorch_cos_sim(term_embedding, summary_embeddings)
                    best_score = similarities.max().item()
                    term_scores.append(best_score)

                semantic_preservation = sum(term_scores) / len(term_scores) if term_scores else 0.0
            except Exception as e:
                print(f"Error calculating semantic preservation: {e}")
                semantic_preservation = 0.5  # Default value

        # Calculate overall compliance score
        if self.model:
            compliance_score = 0.4 * term_preservation + 0.6 * semantic_preservation
        else:
            compliance_score = term_preservation

        # Generate compliance report
        report = {
            'term_preservation_ratio': term_preservation,
            'semantic_preservation_score': semantic_preservation,
            'overall_compliance_score': compliance_score,
            'preserved_terms': list(set(all_original_terms) & set(all_summary_terms)),
            'missing_terms': list(set(all_original_terms) - set(all_summary_terms)),
            'compliance_level': self._get_compliance_level(compliance_score),
            'recommendations': self._generate_recommendations(compliance_score,
                                                            list(set(all_original_terms) - set(all_summary_terms)))
        }

        return report

    def _get_compliance_level(self, score):
        if score >= 0.85:
            return "HIGH"
        elif score >= 0.6:
            return "MODERATE"
        else:
            return "LOW"

    def _generate_recommendations(self, score, missing_terms):
        if score >= 0.85:
            return ["The summary adequately preserves legal terms and context."]

        recommendations = ["Consider revising the summary to include the following missing terms:"]
        for term in missing_terms[:5]:  # Limit to top 5 missing terms
            recommendations.append(f"- '{term}'")

        if score < 0.6:
            recommendations.append("The summary may significantly alter the legal meaning. Consider manual review.")

        return recommendations

# Modify the LegalDocumentProcessor to use the multi-model approach
class LegalDocumentProcessor:
    def __init__(self):
        print("Initializing Legal Document Processor...")
        self.preprocessor = LegalPreprocessor()
        print("Legal preprocessor initialized")
        self.multi_model_summarizer = MultiModelSummarizer()
        print("Multi-model summarizer initialized")
        self.compliance_checker = LegalComplianceChecker()
        print("Legal compliance checker initialized")

        # For enhancing the final summary with features from the original models
        try:
            self.semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
            print("Semantic model initialized")
        except Exception as e:
            print(f"Error loading semantic model: {e}")
            self.semantic_model = None

    def _enhance_summary(self, original_text, summary):
        """Enhance the summary with additional features."""
        try:
            if not self.semantic_model:
                return summary

            # Extract key sentences from original text
            original_sentences = sent_tokenize(original_text)
            summary_sentences = sent_tokenize(summary)

            # If summary is too short, try to add important sentences from original
            if len(summary_sentences) < 3 and len(original_sentences) > 5:
                # Encode sentences
                orig_embeddings = self.semantic_model.encode(original_sentences)
                summary_embeddings = self.semantic_model.encode(summary_sentences)

                # Find sentences in original that are important but not in summary
                important_sentences = []
                for i, orig_sent_emb in enumerate(orig_embeddings):
                    max_sim = max([util.pytorch_cos_sim(orig_sent_emb, summ_emb).item()
                                 for summ_emb in summary_embeddings], default=0)

                    # If sentence is not similar to any in summary but contains legal terms
                    if max_sim < 0.7 and self.compliance_checker.terms_processor.identify_legal_terms(original_sentences[i]):
                        important_sentences.append((original_sentences[i], max_sim))

                # Add up to 2 important sentences
                if important_sentences:
                    important_sentences.sort(key=lambda x: x[1])  # Sort by similarity (ascending)
                    for sent, _ in important_sentences[:2]:
                        if sent not in summary:
                            summary += " " + sent

            return summary
        except Exception as e:
            print(f"Error enhancing summary: {e}")
            return summary

    def process_document(self, text):
        try:
            print("Processing document...")
            # Preprocess the text
            preprocessed_text = self.preprocessor.extract_legal_sections(text)
            print("Text preprocessed")

            # Generate summary using the multi-model approach
            multi_model_summary = self.multi_model_summarizer.summarize(preprocessed_text)
            print("Multi-model summary generated")

            # Enhance the summary with additional features
            enhanced_summary = self._enhance_summary(text, multi_model_summary)
            print("Summary enhanced")

            # Calculate evaluation metrics
            metrics = calculate_metrics(text, enhanced_summary)
            print("Metrics calculated")

            # Calculate coherence
            coherence = 0.0
            if self.semantic_model:
                try:
                    text_embedding = self.semantic_model.encode(text)
                    summary_embedding = self.semantic_model.encode(enhanced_summary)
                    coherence = util.pytorch_cos_sim(text_embedding, summary_embedding).item()
                except Exception as e:
                    print(f"Error calculating coherence: {e}")
            print("Coherence analyzed")

            # Check legal compliance
            compliance_report = self.compliance_checker.check_compliance(text, enhanced_summary)
            print("Legal compliance check completed")

            return {
                'original_text': text,
                'preprocessed_text': preprocessed_text,
                'multi_model_summary': multi_model_summary,
                'final_summary': enhanced_summary,
                'coherence_score': coherence,
                'evaluation_metrics': metrics,
                'legal_compliance': compliance_report
            }
        except Exception as e:
            print(f"Error processing document: {e}")
            return {
                'original_text': text,
                'preprocessed_text': text,
                'multi_model_summary': "Error generating summary",
                'final_summary': "Error processing document",
                'coherence_score': 0,
                'evaluation_metrics': {
                    "ROUGE-1": 0, "ROUGE-2": 0, "ROUGE-L": 0,
                    "BERTScore-F1": 0, "BLEU": 0
                },
                'legal_compliance': {
                    'term_preservation_ratio': 0,
                    'semantic_preservation_score': 0,
                    'overall_compliance_score': 0,
                    'preserved_terms': [],
                    'missing_terms': [],
                    'compliance_level': "ERROR",
                    'recommendations': ["Error processing legal compliance"]
                }
            }

# Get user input directly during runtime
def get_user_input():
    print("\n======================================================")
    print("LEGAL DOCUMENT SUMMARIZER")
    print("======================================================")
    print("\nPlease enter or paste the legal text you want to summarize.")
    print("Type 'END' on a new line when you're finished:")

    lines = []
    while True:
        line = input()
        if line.strip() == "END":
            break
        lines.append(line)

    text = "\n".join(lines)

    if not text.strip():
        print("No text entered. Please run the program again and enter some text.")
        return None

    return text

# Updated function to process text and display results with multi-model approach
def process_legal_text(text):
    if text is None or not text.strip():
        print("\nNo text to process. Please enter some text when prompted.")
        return None

    try:
        print("\nCreating legal document processor...")
        processor = LegalDocumentProcessor()
        print("Processing document...")
        results = processor.process_document(text)

        # Pretty print results
        print("\n======================================================")
        print("LEGAL DOCUMENT SUMMARIZATION RESULTS")
        print("======================================================\n")

        print(f"ORIGINAL TEXT ({len(results['original_text'])} chars, ~{len(results['original_text'].split())} words):")
        print("-" * 60)
        # Print wrapped text for better readability
        print(textwrap.fill(results['original_text'][:500], width=80))
        if len(results['original_text']) > 500:
            print("...(truncated for display)...")

        print("\nPREPROCESSED TEXT:")
        print("-" * 60)
        preprocessed_preview = results['preprocessed_text'][:300]
        print(textwrap.fill(preprocessed_preview, width=80))
        if len(results['preprocessed_text']) > 300:
            print("...(truncated for display)...")

        print("\nMULTI-MODEL SUMMARY:")
        print("-" * 60)
        print(textwrap.fill(results['multi_model_summary'], width=80))

        print("\nENHANCED FINAL SUMMARY:")
        print("-" * 60)
        print(textwrap.fill(results['final_summary'], width=80))

        print("\nEVALUATION METRICS:")
        print("-" * 60)
        print(f"COHERENCE SCORE: {results['coherence_score']:.4f}")
        for metric, value in results['evaluation_metrics'].items():
            print(f"{metric}: {value:.4f}")

        # Display legal compliance information
        print("\nLEGAL COMPLIANCE EVALUATION:")
        print("-" * 60)
        compliance = results['legal_compliance']
        print(f"COMPLIANCE LEVEL: {compliance['compliance_level']}")
        print(f"Term Preservation: {compliance['term_preservation_ratio']:.4f}")
        print(f"Semantic Preservation: {compliance['semantic_preservation_score']:.4f}")
        print(f"Overall Compliance Score: {compliance['overall_compliance_score']:.4f}")

        print("\nPreserved Legal Terms:")
        if compliance['preserved_terms']:
            for term in compliance['preserved_terms']:
                print(f"- {term}")
        else:
            print("- None found")

        print("\nMissing Legal Terms:")
        if compliance['missing_terms']:
            for term in compliance['missing_terms']:
                print(f"- {term}")
        else:
            print("- None missing")

        print("\nRecommendations:")
        for rec in compliance['recommendations']:
            print(f"{rec}")

        print("\nDone! You can now run the program again if you want to process another document.")

        return results

    except Exception as e:
        print(f"An error occurred in execution: {e}")
        return None

# Main function
def main():
    print("Welcome to the Legal Document Summarizer!")

    # Get input directly from user
    legal_text = get_user_input()

    # Process the text
    if legal_text:
        process_legal_text(legal_text)

# Run this directly
if __name__ == "__main__":
    main()

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


Welcome to the Legal Document Summarizer!

LEGAL DOCUMENT SUMMARIZER

Please enter or paste the legal text you want to summarize.
Type 'END' on a new line when you're finished:
Employment Contract  THIS EMPLOYMENT AGREEMENT (the "Agreement") is made and entered into as of this 1st day of April, 2025, by and between ABC Corporation, a company duly registered under the laws of India, having its principal place of business at Mumbai (hereinafter referred to as the "Employer"), and John Doe, an individual residing in New Delhi (hereinafter referred to as the "Employee").  1. Employment Terms 1.1 The Employee agrees to be employed as a Software Engineer and shall perform duties assigned by the Employer. 1.2 The Employee shall serve a probationary period of six (6) months.  2. Compensation & Benefits 2.1 The Employee shall receive a monthly salary of ₹80,000, subject to statutory deductions. 2.2 The Employee is entitled to health insurance, paid leave of 20 days per year, and participation i

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Metrics calculated
Coherence analyzed
Error calculating semantic preservation: 
**********************************************************************
  Resource punkt_tab not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')
  
  For more information see: https://www.nltk.org/data.html

  Attempted to load tokenizers/punkt_tab/english/

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************

Legal compliance check completed

LEGAL DOCUMENT SUMMARIZATION RESULTS

ORIGINAL TEXT (1629 chars, ~258 words):
------------------------------------------------------------
Employment Contract  THIS EMPLOYMENT AGREEMENT (the "Agreement") is made and
entered into as of this 1st da